# DataMate — Làn D: chuẩn hoá + helper code-agent

Làn **1D** của planning DataMate. Kiến trúc `helper`: model viết code trên bảng đã làm sạch (cache profiler) + thư viện helper (kỳ thời gian, lọc đơn, xếp hạng, quy đổi).

**Cách dùng:** Add input dataset `thanhkhil/zaloai-datamate-dataset`, bật GPU T4 x2 + Internet, sửa `CONFIG` nếu cần rồi *Run All*.
- Mặc định chạy toàn bộ **public_test** và ghi `/kaggle/working/submission.jsonl` (nộp file này lên Public Leaderboard).
- Đo trên train: `split="train"`, `subset="dev_tune"` (hoặc `final_holdout`), sẽ có `merge_report.json` với PA theo shop / theo kiểu đáp án.
- Hết giờ giữa chừng: chạy lại version mới, attach output version cũ làm input và điền thư mục đó vào `resume_from_dirs` (cùng config).
- `finalize=True`: câu chưa xử lý xong được ghi `abstain` (0 điểm) để luôn có file nộp hợp lệ.

In [ ]:
CONFIG = {
  "track": "datamate",
  "experiment": "dm-1D-helper-qwen3-4b",
  "parent_run": None,
  "split": "public_test",
  "subset": "all",
  "manifest_seed": 2026,
  "limit": None,
  "mode": "helper",
  "input_roots": [
    "/kaggle/input"
  ],
  "out_dir": "/kaggle/working/dm_helper",
  "cache_dir": "/kaggle/working/dm_cache",
  "llm": {
    "backend": "auto",
    "model": "Qwen/Qwen3-4B-Instruct-2507",
    "revision": None,
    "dtype": "half",
    "max_model_len": 16384,
    "gpu_memory_utilization": 0.9,
    "enable_thinking": False,
    "load_in_4bit": False,
    "batch_size": 4,
    "seed": 0
  },
  "agent": {
    "max_new_tokens": 1536,
    "exec_timeout_s": 60,
    "exec_mem_mb": 6144,
    "exec_workers": 1,
    "repair": True,
    "rules_k": 8,
    "context_chars": 9000
  },
  "batch_size": 16,
  "cpu_threads": 1,
  "seed": 0,
  "wallclock_s": 37800.0,
  "finalize": True,
  "install_vllm": True,
  "resume_from_dirs": []
}

CONFIG_PRESET = "public"  # public | baseline_dev | raw_dev
if CONFIG_PRESET in ("baseline_dev", "raw_dev"):
    CONFIG.update(preset=CONFIG_PRESET,strict_dev=True,split="train",subset="dev_tune",finalize=False,
                  mode="raw" if CONFIG_PRESET=="raw_dev" else "helper",
                  out_dir="/kaggle/working/dm_raw_baseline" if CONFIG_PRESET=="raw_dev" else "/kaggle/working/dm_baseline")

# Overrides for the new dev presets, applied after the preset is resolved.
DEV_OPTIONS = {}  # execution_id, resume_from_dirs, preflight_dirs, control_dir; DataMate: preflight_only


In [ ]:
import os, sys, json, time, glob, shutil, subprocess
from pathlib import Path
SESSION_STARTED_EPOCH = time.time()
WORK = Path("/kaggle/working")
LIB = WORK / "lib"
(LIB / "dmlib").mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(LIB))
OUT = Path(CONFIG["out_dir"]); OUT.mkdir(parents=True, exist_ok=True)

def sh(cmd):
    print("$", cmd); r = subprocess.run(cmd, shell=True, capture_output=True, text=True); print((r.stdout + r.stderr)[-3000:]); return r.returncode

## Thư viện `dmlib` (nhúng sẵn)

In [ ]:
%%writefile /kaggle/working/lib/dmlib/__init__.py
"""DataMate library embedded into the Kaggle notebooks (see build_notebooks.py)."""

In [ ]:
%%writefile /kaggle/working/lib/dmlib/common.py
"""Run plumbing shared by the DataMate notebooks: durable JSONL shards with resume, stable sharding, config hashes,
atomic writes, input discovery under /kaggle/input and environment info."""
from __future__ import annotations

import hashlib
import json
import os
import platform
import subprocess
import time
from pathlib import Path


def dumps(obj) -> str:
    return json.dumps(obj, ensure_ascii=False, separators=(",", ":"), allow_nan=False, default=str)


def load_jsonl(path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def read_jsonl_tolerant(path) -> tuple[list[dict], int]:
    """Rows of a JSONL file written by a process that may have died mid-line. A broken *last* line is dropped
    (returns how many were dropped); a broken line in the middle means the file is corrupt -> ValueError."""
    rows, lines = [], Path(path).read_text(encoding="utf-8").split("\n")
    while lines and not lines[-1].strip():
        lines.pop()
    dropped = 0
    for i, line in enumerate(lines):
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            if i == len(lines) - 1:
                dropped = 1
            else:
                raise ValueError(f"{path}: corrupt line {i + 1}")
    return rows, dropped


def atomic_write_text(path, text: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def atomic_write_json(path, obj) -> None:
    atomic_write_text(path, json.dumps(obj, ensure_ascii=False, indent=1, default=str))


def sha256_file(path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def config_hash(cfg: dict, ignore=()) -> str:
    """Hash of the settings that change results (shard/output/resume keys go in `ignore`)."""
    keep = {k: v for k, v in cfg.items() if k not in ignore}
    return hashlib.sha256(json.dumps(keep, sort_keys=True, default=str).encode()).hexdigest()[:16]


def shard_of(key: str, num_shards: int) -> int:
    """Stable shard index (does not depend on PYTHONHASHSEED or on the order of the inputs)."""
    if num_shards <= 1:
        return 0
    return int(hashlib.md5(key.encode()).hexdigest(), 16) % num_shards


class DurableJsonl:
    """Append-only result file of one worker. First line is a header with the run hash; every record is flushed and
    fsync'ed. On reopen: a truncated last line is dropped, the file is rewritten clean, and a header with another run
    hash is refused (never mix results of different configs). done_ids() = ids whose latest record is 'completed'."""

    def __init__(self, path, run_hash: str, metadata: dict | None = None):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.run_hash = run_hash
        self.records: dict[str, dict] = {}
        if self.path.exists() and self.path.stat().st_size:
            rows, dropped = read_jsonl_tolerant(self.path)
            header = rows[0].get("_header") if rows else None
            if not header or header.get("run_hash") != run_hash:
                raise RuntimeError(f"{self.path} belongs to run {header and header.get('run_hash')}, not {run_hash}")
            for r in rows[1:]:
                self.records[r["id"]] = r
            if dropped:
                atomic_write_text(self.path, "\n".join(dumps(r) for r in rows) + "\n")
        else:
            atomic_write_text(self.path, dumps({"_header": {"run_hash": run_hash, "created": time.time(), **(metadata or {})}}) + "\n")
        self._f = open(self.path, "a", encoding="utf-8")

    def done_ids(self) -> set[str]:
        return {i for i, r in self.records.items() if r.get("status") == "completed"}

    def append(self, rec: dict) -> None:
        previous = self.records.get(rec["id"])
        if previous and previous.get("status") == "completed":
            if previous != rec:
                raise ValueError(f"conflicting completed record: {rec['id']}")
            return
        self._f.write(dumps(rec) + "\n")
        self._f.flush()
        os.fsync(self._f.fileno())
        self.records[rec["id"]] = rec

    def close(self) -> None:
        if not self._f.closed:
            self._f.close()


def find_split_dir(split: str, roots=("/kaggle/input",), marker: str = "questions.jsonl", max_depth: int = 6) -> Path:
    """Directory `<...>/<split>` that holds `marker` (datasets get mounted as /kaggle/input/<slug>/<split>/<split>/...)."""
    for root in map(Path, roots):
        if not root.exists():
            continue
        base_depth = len(root.parts)
        for dirpath, dirnames, filenames in os.walk(root):
            p = Path(dirpath)
            if len(p.parts) - base_depth > max_depth:
                dirnames[:] = []
                continue
            if p.name == split and marker in filenames:
                return p
    raise FileNotFoundError(f"no {split}/{marker} under {list(roots)}")


def env_info() -> dict:
    info = {"python": platform.python_version(), "platform": platform.platform(), "cpus": os.cpu_count()}
    try:
        info["gpus"] = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True, timeout=20
        ).stdout.strip().splitlines()
    except Exception:
        info["gpus"] = []
    for mod in ("torch", "transformers", "vllm", "pandas"):
        try:
            info[mod] = __import__(mod).__version__
        except Exception:
            pass
    try:
        with open("/proc/meminfo") as f:
            info["mem_total_kb"] = int(f.readline().split()[1])
    except Exception:
        pass
    return info


def set_thread_env(threads: int = 1) -> None:
    """One BLAS/OpenMP thread per process (Kaggle has few CPUs)."""
    for k in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        os.environ.setdefault(k, str(threads))


class Deadline:
    """Wall-clock budget of a session: stop taking new work at `stop_after_s`."""

    def __init__(self, stop_after_s: float, end_epoch=None, reserve_s=0):
        self.t0 = time.time()
        self.stop_after_s = min(stop_after_s, end_epoch - self.t0 if end_epoch is not None else stop_after_s) - reserve_s

    def elapsed(self) -> float:
        return time.time() - self.t0

    def expired(self) -> bool:
        return self.elapsed() >= self.stop_after_s

def session_deadline(cfg):
    return Deadline(float(cfg.get("session_wallclock_s", cfg.get("wallclock_s", 14400))),
                    cfg.get("session_deadline_epoch"), cfg.get("export_reserve_s", 0))


def can_start(deadline, estimate_s=0):
    return deadline.elapsed() + max(float(estimate_s), 0) < deadline.stop_after_s


class MemoryMonitor:
    """Sample this worker's nvidia-smi memory, including backend subprocesses on its visible GPU."""
    def __init__(self):
        import threading
        self.peak_mb = None
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()

    def _poll(self):
        while not self._stop.is_set():
            try:
                raw = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,used_memory", "--format=csv,noheader,nounits"],
                                     capture_output=True, text=True, timeout=3).stdout
                # Main worker + descendants (vLLM engine processes), excluding unrelated workers.
                parents = {os.getpid()}
                proc_parents = {}
                for p in Path("/proc").glob("[0-9]*/status"):
                    try:
                        fields = dict(l.split(":", 1) for l in p.read_text().splitlines() if ":" in l)
                        proc_parents[int(p.parent.name)] = int(fields["PPid"].strip())
                    except (OSError, ValueError, KeyError): pass
                while True:
                    new = {pid for pid, parent in proc_parents.items() if parent in parents} - parents
                    if not new: break
                    parents.update(new)
                used = sum(int(line.split(",")[1]) for line in raw.splitlines()
                           if line.strip() and int(line.split(",")[0]) in parents)
                self.peak_mb = max(self.peak_mb or 0, used)
            except (OSError, ValueError, subprocess.TimeoutExpired): pass
            self._stop.wait(.5)

    def close(self):
        self._stop.set()
        self._thread.join(timeout=4)
        return self.peak_mb


def wait_workers(processes, cfg):
    """Stream log files and heartbeat while polling every worker and respecting export reserve."""
    import signal
    started = time.time()
    deadline = cfg.get('session_deadline_epoch')
    end = deadline - float(cfg.get('export_reserve_s', 0)) if deadline is not None else None
    interval = max(.01, float(cfg.get('heartbeat_s', 15)))
    poll = max(.01, min(float(cfg.get('worker_poll_s', 1)), interval))
    phase = cfg.get('worker_phase', 'inference')
    next_heartbeat = started
    offsets = [0] * len(processes)
    codes = [None] * len(processes)

    def tail(index, log):
        try:
            with open(log.name, 'rb') as reader:
                reader.seek(0, 2)
                size = reader.tell()
                if size <= offsets[index]: return
                maximum = 12000
                offset = max(offsets[index], size - maximum)
                reader.seek(offset)
                content = reader.read(maximum)
                offsets[index] = reader.tell()
            if offset > 0 and offset != size - len(content) and b'\n' in content:
                content = content.split(b'\n', 1)[-1]
            for line in content.decode('utf-8', 'replace').splitlines():
                print(f'[{phase} worker {index}] {line}', flush=True)
        except (OSError, AttributeError):
            pass

    def heartbeat(now):
        states = []
        for index, (process, _) in enumerate(processes):
            state = f'worker{index}:running' if process.poll() is None else f'worker{index}:exit={codes[index]}'
            if phase == 'inference' and cfg.get('out_dir'):
                path = Path(cfg['out_dir']) / f'status.shard{index}.json'
                try:
                    status = json.loads(path.read_text())
                    state += (f" stage={status.get('stage')} done={status.get('completed', 0)}/{status.get('total', '?')}"
                              f" last_update={max(0, now-status.get('updated_epoch', now)):.0f}s ago")
                except (OSError, ValueError): pass
            states.append(state)
        print(f'[{phase} heartbeat] elapsed={now-started:.0f}s | ' + ' | '.join(states), flush=True)

    try:
        while any(code is None for code in codes):
            now = time.time()
            for index, (process, log) in enumerate(processes):
                tail(index, log)
                if codes[index] is not None: continue
                result = process.poll()
                if result is not None:
                    codes[index] = result
                    print(f'[{phase}] worker {index} exited code={result}', flush=True)
                elif end is not None and now >= end:
                    killed = False
                    try:
                        os.killpg(process.pid, signal.SIGKILL)
                        killed = True
                    except ProcessLookupError: pass
                    result = process.wait()
                    codes[index] = 124 if killed and result == -signal.SIGKILL else result
                    if codes[index] == 124:
                        print(f'[{phase}] worker {index} stopped at work deadline; checkpoint retained', flush=True)
                    else:
                        print(f'[{phase}] worker {index} exited code={result} at deadline', flush=True)
            if now >= next_heartbeat:
                heartbeat(now)
                next_heartbeat = now + interval
            if any(code is None for code in codes): time.sleep(poll)
        for index, (_, log) in enumerate(processes): tail(index, log)
    finally:
        for process, log in processes:
            try: os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError: pass
            if process.poll() is None: process.wait()
            log.close()
    return codes


In [ ]:
%%writefile /kaggle/working/lib/dmlib/parsers.py
"""Parse the messy cells of shop files: VN money/number strings, dates, quantities with units, order status."""
from __future__ import annotations

import datetime as dt
import functools
import math
import re
import unicodedata

STATUSES = ("hoàn thành", "huỷ", "trả hàng", "đang giao")


def fold(s) -> str:
    """Lower-case, strip Vietnamese diacritics (đ -> d), collapse spaces."""
    if s is None:
        return ""
    return _fold_str(str(s))


@functools.lru_cache(maxsize=200_000)
def _fold_str(s: str) -> str:
    t = unicodedata.normalize("NFD", s.lower()).replace("đ", "d")
    return " ".join("".join(c for c in t if not unicodedata.combining(c)).split())


def _is_missing(v) -> bool:
    return v is None or (isinstance(v, float) and math.isnan(v))


def _plain_number(s: str):
    """'227,262,100' / '1.000.000' / '1,090,000.0000' / '16 076' / '1,45' / '0.4' -> float, or None."""
    s = s.strip().replace(" ", " ")
    if not s:
        return None
    sign = -1 if s.startswith("-") else 1
    s = s.lstrip("+-").strip()
    if re.fullmatch(r"\d{1,3}( \d{3})+", s):
        return sign * float(s.replace(" ", ""))
    if not re.fullmatch(r"[\d.,]+", s) or not re.search(r"\d", s):
        return None
    dots, commas = s.count("."), s.count(",")
    if dots and commas:
        # the separator that comes last is the decimal one
        dec = "." if s.rfind(".") > s.rfind(",") else ","
        thou = "," if dec == "." else "."
        s = s.replace(thou, "").replace(dec, ".")
    elif dots > 1 or commas > 1:
        sep = "." if dots else ","
        if not all(len(g) == 3 for g in s.split(sep)[1:]):
            return None
        s = s.replace(sep, "")
    elif dots == 1 or commas == 1:
        sep = "." if dots else ","
        head, tail = s.split(sep)
        # one group of exactly 3 digits after the separator is a thousands separator (VN money style)
        s = head + tail if len(tail) == 3 and head and head != "0" else head + "." + tail
    try:
        return sign * float(s)
    except ValueError:
        return None


def _as_int_if_whole(x: float):
    return int(round(x)) if abs(x - round(x)) < 1e-9 else x


def parse_number(v):
    """Number cell -> int/float or None. Handles '144.000đ', '750k', '12tr460', '1,45tr', '2 triệu'."""
    if _is_missing(v) or isinstance(v, bool):
        return None
    if isinstance(v, (int, float)):
        return _as_int_if_whole(float(v))
    s = fold(v).replace("vnd", "").replace("dong", "").strip()
    s = re.sub(r"(?<=\d)\s*d$", "", s).strip()
    if not s:
        return None
    m = re.fullmatch(r"(-?[\d.,]+)\s*(tr|trieu)\s*(\d{1,3})?", s)
    if m:
        base = _plain_number(m.group(1))
        if base is None:
            return None
        extra = int(m.group(3).ljust(3, "0")) * 1000 if m.group(3) else 0
        return _as_int_if_whole(base * 1_000_000 + (extra if base >= 0 else -extra))
    m = re.fullmatch(r"(-?[\d.,]+)\s*(k|nghin|ngan)", s)
    if m:
        base = _plain_number(m.group(1))
        return None if base is None else _as_int_if_whole(base * 1000)
    x = _plain_number(s)
    return None if x is None else _as_int_if_whole(x)


_THG = re.compile(r"(\d{1,2})\s*thg\s*(\d{1,2})\s*(\d{4})")
_DMY = re.compile(r"(\d{1,2})[/\-.](\d{1,2})[/\-.](\d{4})")
_YMD = re.compile(r"(\d{4})-(\d{1,2})-(\d{1,2})")
_DM = re.compile(r"(\d{1,2})[/\-.](\d{1,2})")


def _safe_date(y, m, d):
    try:
        return dt.date(int(y), int(m), int(d))
    except ValueError:
        return None


def parse_date(v, default_year: int | None = None):
    """Date cell -> datetime.date or None. A 'd/m' cell without a year takes `default_year` (the sheet's year)."""
    if _is_missing(v):
        return None
    if isinstance(v, dt.datetime):
        return v.date()
    if isinstance(v, dt.date):
        return v
    if hasattr(v, "to_pydatetime"):  # pandas Timestamp
        return v.to_pydatetime().date()
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        # Excel serial day number (1899-12-30 epoch); only plausible years 1990..2060
        if 32874 <= v <= 58440:
            return dt.date(1899, 12, 30) + dt.timedelta(days=int(v))
        return None
    s = str(v).strip()
    for rx, order in ((_YMD, "ymd"), (_THG, "dmy"), (_DMY, "dmy")):
        m = rx.search(s)
        if m:
            a, b, c = m.groups()
            return _safe_date(a, b, c) if order == "ymd" else _safe_date(c, b, a)
    m = _DM.fullmatch(s)
    if m and default_year:
        return _safe_date(default_year, m.group(2), m.group(1))
    return None


def parse_qty(v):
    """'3 m' -> (3, 'm'); '1,5 kg' -> (1.5, 'kg'); 2 -> (2, None)."""
    if _is_missing(v):
        return (None, None)
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return (_as_int_if_whole(float(v)), None)
    m = re.fullmatch(r"\s*(-?[\d.,]+)\s*(.*?)\s*", str(v))
    if not m:
        return (None, None)
    num = m.group(1)
    if "," in num and "." not in num and len(num.split(",")[1]) != 3:
        num = num.replace(",", ".")
    q = parse_number(num)
    return (q, m.group(2) or None) if q is not None else (None, None)


_STATUS_KEYS = [
    ("huỷ", ("huy", "cancel", "cancelled", "canceled", "bom hang", "khach bom")),
    ("trả hàng", ("tra hang", "tra lai", "hoan hang", "return", "doi tra", "tra mot phan", "tra 1 phan", "returned", "refund", "refunded")),
    ("đang giao", ("dang giao", "dang ship", "shipping", "dang van chuyen", "cho giao", "dang chuyen", "delivering", "in transit", "shipped")),
    ("hoàn thành", ("hoan thanh", "da xong", "xong", "done", "complete", "completed", "thanh cong", "da giao", "ht", "ok", "da thanh toan")),
]


_STATUS_RX = [
    (status, re.compile("(^|[^a-z])(" + "|".join(re.escape(k) for k in keys) + ")([^a-z]|$)")) for status, keys in _STATUS_KEYS
]


def norm_status(v):
    """Map a status / note cell to one of the four rulebook statuses, or None if it names none."""
    f = fold(v)
    if not f:
        return None
    for status, rx in _STATUS_RX:
        if rx.search(f):
            return status
    return None

In [ ]:
%%writefile /kaggle/working/lib/dmlib/periods.py
"""Rulebook §2 time periods. Every period is an inclusive (start, end) pair of datetime.date."""
from __future__ import annotations

import calendar as _cal
import datetime as dt

import pandas as pd

Period = tuple  # (dt.date, dt.date), both inclusive


def to_date(d) -> dt.date:
    if isinstance(d, dt.datetime):
        return d.date()
    if isinstance(d, dt.date):
        return d
    return pd.Timestamp(d).date()


def _month_end(y: int, m: int) -> dt.date:
    return dt.date(y, m, _cal.monthrange(y, m)[1])


def day(d) -> Period:
    d = to_date(d)
    return (d, d)


def yesterday(as_of) -> Period:
    return day(to_date(as_of) - dt.timedelta(days=1))


def this_week(as_of) -> Period:
    """Monday of the week holding as_of .. as_of."""
    d = to_date(as_of)
    return (d - dt.timedelta(days=d.weekday()), d)


def last_week(as_of) -> Period:
    """The whole previous Monday..Sunday week."""
    monday = this_week(as_of)[0]
    return (monday - dt.timedelta(days=7), monday - dt.timedelta(days=1))


def last_n_days(as_of, n: int) -> Period:
    """N consecutive days ending today (today included)."""
    d = to_date(as_of)
    return (d - dt.timedelta(days=n - 1), d)


def this_month(as_of) -> Period:
    d = to_date(as_of)
    return (d.replace(day=1), d)


def same_days_prev_month(as_of) -> Period:
    """Day 1 .. same day number of the previous month (or that month's end if it has no such day)."""
    d = to_date(as_of)
    y, m = (d.year, d.month - 1) if d.month > 1 else (d.year - 1, 12)
    end = min(d.day, _cal.monthrange(y, m)[1])
    return (dt.date(y, m, 1), dt.date(y, m, end))


def prev_month_full(as_of) -> Period:
    d = to_date(as_of)
    y, m = (d.year, d.month - 1) if d.month > 1 else (d.year - 1, 12)
    return (dt.date(y, m, 1), _month_end(y, m))


def month(m: int, year: int | None = None, as_of=None) -> Period:
    """Whole month M. Without a year: this year unless that month starts after as_of, then last year."""
    if year is None:
        d = to_date(as_of)
        year = d.year if dt.date(d.year, m, 1) <= d else d.year - 1
    return (dt.date(year, m, 1), _month_end(year, m))


def quarter(q: int, year: int | None = None, as_of=None) -> Period:
    """Calendar quarter Q (Q1 = Jan..Mar). Without a year: same roll-back rule as month()."""
    first = 3 * (q - 1) + 1
    if year is None:
        d = to_date(as_of)
        year = d.year if dt.date(d.year, first, 1) <= d else d.year - 1
    return (dt.date(year, first, 1), _month_end(year, first + 2))


def year_to_date(as_of) -> Period:
    d = to_date(as_of)
    return (dt.date(d.year, 1, 1), d)


def whole_year(year: int) -> Period:
    return (dt.date(year, 1, 1), dt.date(year, 12, 31))


def prev_period(p: Period) -> Period:
    """The period of the same length right before p."""
    start, end = to_date(p[0]), to_date(p[1])
    n = (end - start).days + 1
    return (start - dt.timedelta(days=n), start - dt.timedelta(days=1))


def clip_to_asof(p: Period, as_of) -> Period:
    """Records after today have not happened yet."""
    return (to_date(p[0]), min(to_date(p[1]), to_date(as_of)))


def _lunar_to_solar(calendar: pd.DataFrame, lday: int, lmonth: int, lyear: int) -> dt.date:
    cal = calendar
    leap_col = "lunar_leap" if "lunar_leap" in cal.columns else ("is_leap_month" if "is_leap_month" in cal.columns else None)
    mask = (cal["lunar_day"].astype(int) == lday) & (cal["lunar_month"].astype(int) == lmonth) & (cal["lunar_year"].astype(int) == lyear)
    if leap_col is not None:
        mask &= ~cal[leap_col].astype(str).str.lower().isin(["true", "1"])
    hit = cal.loc[mask, "date"]
    if hit.empty:
        raise KeyError(f"lunar {lday}/{lmonth}/{lyear} not in calendar.csv")
    return to_date(sorted(hit.astype(str))[0])


def tet_range(year: int, calendar: pd.DataFrame) -> Period:
    """Tết of year Y: solar date of 23 tháng Chạp (lunar year Y-1) .. solar date of mùng 10 tháng Giêng (lunar year Y)."""
    return (_lunar_to_solar(calendar, 23, 12, year - 1), _lunar_to_solar(calendar, 10, 1, year))


def before_tet_range(year: int, calendar: pd.DataFrame) -> Period:
    """Same number of days right before 23 tháng Chạp."""
    return prev_period(tet_range(year, calendar))


def shift_of(ts) -> str | None:
    """Ca sáng 06:00–13:59, ca chiều 14:00–21:59 (by order creation time); other hours: None."""
    if ts is None or pd.isna(ts):
        return None
    h = pd.Timestamp(ts).hour
    if 6 <= h <= 13:
        return "sáng"
    if 14 <= h <= 21:
        return "chiều"
    return None


def count_weekday(p: Period, weekday: int) -> int:
    """Number of days with that weekday (Monday=0) in p, days without orders included."""
    days = pd.date_range(to_date(p[0]), to_date(p[1]), freq="D")
    return int((days.day_of_week == weekday).sum())


def in_range(dates: pd.Series, p: Period) -> pd.Series:
    """Boolean mask: date (or datetime) within the inclusive period."""
    d = pd.to_datetime(dates, errors="coerce").dt.normalize()
    return (d >= pd.Timestamp(to_date(p[0]))) & (d <= pd.Timestamp(to_date(p[1])))


def overlap_status(p: Period, data_range: Period) -> str:
    """'outside' (no day of p has data -> unanswerable), 'partial' (compute on the covered part) or 'inside'."""
    s, e = to_date(p[0]), to_date(p[1])
    ds, de = to_date(data_range[0]), to_date(data_range[1])
    if e < ds or s > de:
        return "outside"
    if s < ds or e > de:
        return "partial"
    return "inside"


def pct_change(cur, prev, ndigits: int = 2):
    """Percent change vs the previous period (negative when it fell); None when prev is 0."""
    if prev in (0, None) or cur is None:
        return None
    return round((cur - prev) / prev * 100, ndigits)

In [ ]:
%%writefile /kaggle/working/lib/dmlib/answer.py
"""Answer formatting (rulebook §9: one format per type) and the local scorer used on train/dev.

Scorer assumptions (not confirmed against the organisers' scorer): the predicted `type` must equal the gold type;
integer references match exactly; other numbers round to 2 decimals; percent is compared after rounding both sides to 2 decimals; strings match the gold answer or
any alias after norm_string; table columns are matched by normalised name, else by position.
"""
from __future__ import annotations

import math
import re
import unicodedata
from decimal import ROUND_HALF_UP, Decimal

from .parsers import parse_number

TYPES = ("number", "percent", "string", "date", "boolean", "list", "table", "unanswerable", "abstain")


class AnswerError(ValueError):
    """The value cannot be written in the single format its type allows."""


def round_half_up(x: float, ndigits: int = 0):
    q = Decimal(1).scaleb(-ndigits)
    r = Decimal(repr(float(x))).quantize(q, rounding=ROUND_HALF_UP)
    return int(r) if ndigits == 0 else float(r)


def norm_string(s) -> str:
    """NFC, lower-case, every char that is not a letter/digit/space -> space, collapse spaces. Diacritics kept."""
    t = unicodedata.normalize("NFC", str(s)).lower()
    t = "".join(c if (c.isalnum() or c.isspace()) else " " for c in t)
    return " ".join(t.split())


def _finite_number(v):
    if isinstance(v, bool):
        raise AnswerError("boolean is not a number")
    if hasattr(v, "item") and not isinstance(v, (list, dict, str)):  # numpy scalar
        v = v.item()
    if isinstance(v, str):
        p = parse_number(v)
        if p is None:
            raise AnswerError(f"not a number: {v!r}")
        v = p
    if not isinstance(v, (int, float)) or isinstance(v, bool):
        raise AnswerError(f"not a number: {v!r}")
    if isinstance(v, float) and not math.isfinite(v):
        raise AnswerError("NaN/Infinity is not valid JSON")
    return v


def _fmt_number(v, money: bool = False):
    v = _finite_number(v)
    if money:
        return round_half_up(v, 0)
    if isinstance(v, int):
        return v
    r = round_half_up(v, 2)
    return int(r) if r == int(r) else r


def _fmt_scalar_cell(v):
    if isinstance(v, str):
        return v
    return _fmt_number(v)


_DATE = re.compile(r"\d{4}-\d{2}(-\d{2})?")


def format_value(typ: str, value, money: bool = False):
    if typ == "number":
        return _fmt_number(value, money)
    if typ == "percent":
        return _fmt_number(round_half_up(_finite_number(value), 2))
    if typ == "string":
        if not isinstance(value, str) or not value.strip():
            raise AnswerError("string answer must be a non-empty string")
        return unicodedata.normalize("NFC", value.strip())
    if typ == "date":
        if hasattr(value, "strftime"):
            value = value.strftime("%Y-%m-%d")
        if not isinstance(value, str) or not _DATE.fullmatch(value):
            raise AnswerError(f"date must be YYYY-MM-DD or YYYY-MM: {value!r}")
        import datetime
        try:
            if len(value)==7:datetime.date.fromisoformat(value+"-01")
            else:datetime.date.fromisoformat(value)
        except ValueError:raise AnswerError(f"invalid calendar date: {value!r}")
        return value
    if typ == "boolean":
        if hasattr(value, "item") and not isinstance(value, (bool, str)):
            value = value.item()
        if not isinstance(value, bool):
            raise AnswerError("boolean must be true/false")
        return value
    if typ == "list":
        if isinstance(value, tuple):
            value = list(value)
        if not isinstance(value, list):
            raise AnswerError("list answer must be a JSON array")
        return [_fmt_scalar_cell(x) for x in value]
    if typ == "table":
        if not isinstance(value, dict) or not isinstance(value.get("columns"), list) or not isinstance(value.get("rows"), list):
            raise AnswerError("table must be {'columns': [...], 'rows': [[...]]}")
        cols = [str(c) for c in value["columns"]]
        rows = []
        for r in value["rows"]:
            if not isinstance(r, (list, tuple)) or len(r) != len(cols):
                raise AnswerError("every table row must have one cell per column")
            rows.append([_fmt_scalar_cell(x) for x in r])
        return {"columns": cols, "rows": rows}
    raise AnswerError(f"unknown type {typ!r}")


def format_answer(qid: str, typ: str, value, reason: str | None = None, money: bool = False) -> dict:
    """Submission line for one question. Raises AnswerError if the value does not fit the type's format."""
    if typ == "abstain":
        return {"id": qid, "type": "abstain"}
    if typ == "unanswerable":
        return {"id": qid, "type": "unanswerable", "reason": str(reason or "không có dữ liệu")}
    if typ not in TYPES:
        raise AnswerError(f"unknown type {typ!r}")
    return {"id": qid, "type": typ, "answer": format_value(typ, value, money)}


def validate_record(rec: dict) -> str | None:
    """None if the submission line is well formed, else the reason."""
    if not isinstance(rec, dict) or not isinstance(rec.get("id"), str):
        return "missing id"
    typ = rec.get("type")
    if typ == "abstain":
        return None
    if typ == "unanswerable":
        return None if isinstance(rec.get("reason", ""), str) else "reason must be a string"
    if "answer" not in rec:
        return "missing answer"
    try:
        out = format_value(typ, rec["answer"])
    except AnswerError as e:
        return str(e)
    if typ in ("number", "percent") and not isinstance(rec["answer"], (int, float)):
        return "number must be a JSON number"
    return None if out is not None else "bad answer"


# ---------------------------------------------------------------- scoring
def _num_eq(p, g, percent=False) -> bool:
    if isinstance(p, bool) or not isinstance(p, (int, float)) or isinstance(g, bool) or not isinstance(g, (int, float)):
        return False
    if not math.isfinite(p) or not math.isfinite(g): return False
    if percent:
        return abs(round_half_up(p, 2) - round_half_up(g, 2)) < 1e-6
    if not math.isfinite(p) or not math.isfinite(g): return False
    return p == g if isinstance(g,int) else round_half_up(p,2) == round_half_up(g,2)


def _cell_eq(p, g, aliases=()) -> bool:
    if isinstance(g, str):
        return isinstance(p, str) and norm_string(p) in {norm_string(x) for x in (g, *aliases)}
    return _num_eq(p, g)


def _table_eq(p, g) -> bool:
    if not isinstance(p,dict) or not isinstance(p.get('columns'),list) or not isinstance(p.get('rows'),list):return False
    gcols,pcols=[norm_string(c) for c in g['columns']],[norm_string(c) for c in p['columns']]
    if len(gcols)!=len(pcols) or len(set(gcols))!=len(gcols) or len(set(pcols))!=len(pcols):return False
    idx=[pcols.index(c) if c in pcols else None for c in gcols]
    used={i for i in idx if i is not None}
    for i,j in enumerate(idx):
        if j is None:
            available=[k for k in range(len(pcols)) if k not in used]
            idx[i]=i if i in available else available[0];used.add(idx[i])
    try:
        if any(not isinstance(row,list) or len(row)!=len(pcols) for row in p['rows']):return False
        prows=[[row[i] for i in idx] for row in p['rows']]
        grows=g['rows']
        # Rows are a set; duplicates do not change the table's meaning.
        equal=lambda a,b:len(a)==len(b) and all(_cell_eq(x,y) for x,y in zip(a,b))
        return all(any(equal(pr,gr) for pr in prows) for gr in grows) and all(any(equal(pr,gr) for gr in grows) for pr in prows)
    except (TypeError,IndexError,KeyError):return False


def score(pred: dict | None, gold: dict) -> int:
    """+1 correct, 0 abstain / missing / unreadable, -1 wrong."""
    if not pred or pred.get("type") in (None, "abstain"):
        return 0
    ptype, gtype = pred.get("type"), gold["type"]
    if ptype == "unanswerable":
        if not isinstance(pred.get("reason", ""), str):
            return 0
        return 1 if gtype == "unanswerable" else -1
    if ptype != gtype or "answer" not in pred:
        return -1
    p, g, aliases = pred["answer"], gold.get("answer"), gold.get("alias") or []
    if gtype == "number":
        ok = _num_eq(p, g)
    elif gtype == "percent":
        ok = _num_eq(p, g, percent=True)
    elif gtype == "string":
        ok = _cell_eq(p, g, aliases)
    elif gtype == "date":
        ok = isinstance(p, str) and p == g
    elif gtype == "boolean":
        ok = isinstance(p, bool) and p == g
    elif gtype == "list":
        ok = isinstance(p, list) and len(p) == len(g) and all(_cell_eq(a, b) for a, b in zip(p, g))
    elif gtype == "table":
        ok = _table_eq(p, g)
    else:
        ok = False
    return 1 if ok else -1


def penalized_accuracy(preds: dict, golds: dict) -> dict:
    """preds/golds keyed by question id; ids missing from preds count as abstain."""
    s = {qid: score(preds.get(qid), g) for qid, g in golds.items()}
    n = len(s)
    correct = sum(v == 1 for v in s.values())
    wrong = sum(v == -1 for v in s.values())
    return {
        "n": n,
        "correct": correct,
        "wrong": wrong,
        "abstain": n - correct - wrong,
        "pa": (correct - wrong) / n if n else 0.0,
        "accuracy_answered": correct / (correct + wrong) if correct + wrong else None,
        "coverage": (correct + wrong) / n if n else 0.0,
        "per_id": s,
    }

In [ ]:
%%writefile /kaggle/working/lib/dmlib/loader.py
"""Read every table of a shop (xlsx sheets, csv, sqlite) as raw cells, with the header row detected and the source
of every row kept in _file / _sheet / _row. No cleaning here: see profiler.py."""
from __future__ import annotations

import csv
import io
import re
import sqlite3
import warnings
from dataclasses import dataclass
from pathlib import Path

import pandas as pd

from .parsers import parse_date, parse_number

LOADER_VERSION = "1"
LINEAGE = ("_file", "_sheet", "_row")


@dataclass
class RawTable:
    name: str
    file: str
    sheet: str | None
    header_row: int | None  # 0-based row of the header inside the sheet / file
    df: pd.DataFrame


def _is_labelish(v) -> bool:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return False
    if not isinstance(v, str) or not v.strip():
        return False
    return parse_number(v) is None and parse_date(v) is None


def detect_header_row(raw: pd.DataFrame, scan: int = 15) -> int:
    """Row with the most label-like cells among the first `scan` rows; ties go to the later row only when the earlier
    row is a sparse group header (e.g. 'Đơn | Hàng | Tiền' above the real column names)."""
    best, best_score = 0, -1
    for i in range(min(scan, len(raw))):
        score = sum(_is_labelish(v) for v in raw.iloc[i].tolist())
        if score > best_score:
            best, best_score = i, score
    return best


def _unique_names(cells) -> list[str]:
    names, seen = [], {}
    for j, c in enumerate(cells):
        n = str(c).strip() if _is_labelish(c) or (isinstance(c, str) and c.strip()) else f"col_{j}"
        n = re.sub(r"\s+", " ", n)
        if n in seen:
            seen[n] += 1
            n = f"{n}_{seen[n]}"
        else:
            seen[n] = 1
        names.append(n)
    return names


def frame_from_grid(raw: pd.DataFrame, file: str, sheet: str | None, header_row: int | None = None) -> tuple[pd.DataFrame, int]:
    if header_row is None:
        header_row = detect_header_row(raw)
    body = raw.iloc[header_row + 1 :].copy()
    body.columns = _unique_names(raw.iloc[header_row].tolist())
    body = body.dropna(how="all")
    body = body.loc[:, [c for c in body.columns if not (c.startswith("col_") and body[c].isna().all())]]
    body.insert(len(body.columns), "_file", file)
    body.insert(len(body.columns), "_sheet", sheet or "")
    body.insert(len(body.columns), "_row", body.index.to_numpy() + 1)  # 1-based row number in the source grid
    return body.reset_index(drop=True), header_row


def _slug(s: str) -> str:
    return re.sub(r"[^\w\-]+", "_", s, flags=re.UNICODE).strip("_")


def read_csv_grid(path: Path) -> pd.DataFrame:
    raw = path.read_bytes()
    for enc in ("utf-8-sig", "cp1258", "latin-1"):
        try:
            text = raw.decode(enc)
            break
        except UnicodeDecodeError:
            continue
    try:
        delim = csv.Sniffer().sniff(text[:5000], delimiters=",;\t|").delimiter
    except csv.Error:
        delim = ","
    return pd.read_csv(io.StringIO(text), header=None, dtype=str, sep=delim, keep_default_na=False, na_values=[""])


def read_excel_grids(path: Path) -> dict[str, pd.DataFrame]:
    """All sheets as raw grids. calamine is ~10x faster than openpyxl; fall back when it is not installed."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            return pd.read_excel(path, sheet_name=None, header=None, engine="calamine")
        except ImportError:
            return pd.read_excel(path, sheet_name=None, header=None)


def load_shop(shop_dir) -> list[RawTable]:
    shop_dir = Path(shop_dir)
    out: list[RawTable] = []
    for path in sorted(shop_dir.iterdir()):
        suf = path.suffix.lower()
        if suf in (".xlsx", ".xlsm", ".xls"):
            sheets = read_excel_grids(path)
            for sheet, grid in sheets.items():
                if grid.dropna(how="all").empty:
                    continue
                df, h = frame_from_grid(grid, path.name, sheet)
                name = _slug(path.stem) if len(sheets) == 1 and sheet.lower().startswith("sheet") else f"{_slug(path.stem)}__{_slug(sheet)}"
                out.append(RawTable(name, path.name, sheet, h, df))
        elif suf in (".csv", ".tsv", ".txt"):
            grid = read_csv_grid(path)
            if grid.empty:
                continue
            df, h = frame_from_grid(grid, path.name, None)
            out.append(RawTable(_slug(path.stem), path.name, None, h, df))
        elif suf in (".sqlite", ".db", ".sqlite3"):
            con = sqlite3.connect(f"file:{path}?mode=ro", uri=True)
            try:
                for (t,) in con.execute("select name from sqlite_master where type='table'"):
                    df = pd.read_sql_query(f'select * from "{t}"', con)
                    df["_file"], df["_sheet"], df["_row"] = path.name, t, range(1, len(df) + 1)
                    out.append(RawTable(f"{_slug(path.stem)}__{_slug(t)}", path.name, t, None, df))
            finally:
                con.close()
    return out

In [ ]:
%%writefile /kaggle/working/lib/dmlib/profiler.py
"""Lane 0 (CPU): profile every shop once and cache clean tables for the helper code-agent.

For each table: typed columns (VN money/number strings, dates, quantities with units), order-level cells forward-filled
inside an order (rulebook §8), exact duplicate lines of the same order dropped, `_status` normalised to the four
statuses. Raw cells are cached next to the clean ones (lineage kept). Plus README facts (data cut-off, unit
conversions) and a capability profile with three states: present / absent_per_readme / unknown.
Nothing here is a semantic mapping to orders/order_lines yet: that is decided after the first round's error analysis.
"""
from __future__ import annotations

import datetime as dt
import json
import re
import shutil
from pathlib import Path

import pandas as pd

from .common import atomic_write_json, atomic_write_text, sha256_file
from .loader import LINEAGE, LOADER_VERSION, RawTable, load_shop
from .parsers import fold, norm_status, parse_date, parse_number, parse_qty

PROFILER_VERSION = "2"

_ID_HINTS = ("ma ", "ma_", "barcode", "sdt", "dien thoai", "so phieu", "bien so", "so dt", "phone", " id", "code", "imei", "serial")
_ORDER_KEY_HINTS = ("ma don", "so don", "ma hd", "so hd", "ma hoa don", "so hoa don", "ma phieu", "so phieu", "order", "invoice", "don hang", "ma bill", "so bill", "bill")
_CAPABILITIES = {
    "status": ("trang thai", "status", "tinh trang"),
    "customer": ("khach", "customer"),
    "customer_group": ("nhom khach", "loai khach", "hang khach", "phan khuc"),
    "staff": ("nhan vien", "thu ngan", "nguoi ban", "tho ", "staff", "cashier", "nguoi lam"),
    "purchase": ("nhap", "nha cung cap", "ncc", "supplier"),
    "stock": ("ton", "stock", "inventory"),
    "cost": ("gia von", "cost", "gia nhap"),
    "payment_method": ("thanh toan", "hinh thuc", "payment", "pttt", "phuong thuc"),
    "debt": ("no ", "cong no", "con no", "ghi no", "debt", "tra gop", "han no"),
    "expiry": ("han dung", "han su dung", "hsd", "expiry", "het han"),
    "time_of_day": ("gio", "time", "thoi gian"),
    "shipping_fee": ("phi ship", "phi giao", "van chuyen", "shipping"),
    "discount": ("giam gia", "chiet khau", "voucher", "khuyen mai", "discount"),
    "returns": ("tra hang", "tra lai", "hoan"),
}


def parse_readme(text: str) -> dict:
    """Data cut-off date and unit conversions ('X: 1 thùng = 24 lon') written in a shop README."""
    info: dict = {"data_until": None, "conversions": {}}
    m = re.search(r"đến hết (?:ngày )?(\d{1,2})/(\d{1,2})/(\d{4})", text)
    if m:
        info["data_until"] = dt.date(int(m.group(3)), int(m.group(2)), int(m.group(1))).isoformat()
    for line in text.splitlines():
        m = re.match(r"\s*[-*]\s*(.+?):\s*1\s+(.+?)\s*=\s*([\d.,]+)\s*(.+?)\s*$", line)
        if not m:
            continue
        item, big, n, base = m.group(1).strip(), m.group(2).strip(), parse_number(m.group(3)), m.group(4).strip()
        if n is None:
            continue
        conv = info["conversions"].setdefault(item, {})
        conv[big] = n
        conv["_base"] = base
    return info


def _year_hint(*names) -> int | None:
    for n in names:
        m = re.search(r"(20\d{2})", str(n or ""))
        if m:
            return int(m.group(1))
    return None


def map_unique(values: pd.Series, fn) -> pd.Series:
    """values.map(fn), computing fn once per distinct value (shop files repeat the same cells a lot)."""
    keys = values.map(lambda v: (type(v).__name__, v) if not (isinstance(v, float) and pd.isna(v)) else None)
    cache: dict = {}
    out = []
    for k, v in zip(keys, values):
        if k not in cache:
            cache[k] = fn(v)
        out.append(cache[k])
    return pd.Series(out, index=values.index, dtype=object)


def _infer_column(name: str, values: pd.Series, year: int | None) -> dict:
    nn = values.dropna()
    nn = nn[nn.astype(str).str.strip() != ""]
    info = {"kind": "empty" if nn.empty else "text", "non_null": int(len(nn)), "nulls": int(len(values) - len(nn))}
    if nn.empty:
        return info
    fname = " " + fold(name) + " "
    sample = nn if len(nn) <= 3000 else nn.sample(3000, random_state=0)
    idlike = any(h in fname for h in _ID_HINTS)
    n_ok = map_unique(sample, lambda v: parse_number(v) is not None).astype(bool).mean()
    d_ok = map_unique(sample, lambda v: parse_date(v, year) is not None).astype(bool).mean()
    is_dt = sample.map(lambda v: isinstance(v, (pd.Timestamp, dt.datetime)) and (v.hour or v.minute or v.second)).mean()
    q = map_unique(sample, parse_qty)
    q_ok = q.map(lambda x: x[0] is not None).mean()
    q_unit = q.map(lambda x: x[1] is not None).mean()
    if d_ok >= 0.9 and not (n_ok >= 0.9 and sample.map(lambda v: isinstance(v, (int, float))).mean() > 0.5):
        info["kind"] = "datetime" if is_dt >= 0.5 else "date"
    elif n_ok >= 0.9 and not idlike:
        info["kind"] = "number"
    elif q_ok >= 0.9 and q_unit >= 0.2 and not idlike:
        info["kind"] = "qty"
    if info["kind"] == "text":
        vc = nn.astype(str).value_counts()
        info["n_unique"] = int(len(vc))
        info["top"] = {str(k): int(v) for k, v in vc.head(12).items()}
    return info


def _clean_column(values: pd.Series, kind: str, year: int | None):
    if kind == "number":
        return pd.to_numeric(map_unique(values, parse_number), errors="coerce"), None
    if kind == "date":
        return pd.to_datetime(map_unique(values, lambda v: parse_date(v, year)), errors="coerce"), None
    if kind == "datetime":
        def conv(v):
            if isinstance(v, (pd.Timestamp, dt.datetime)):
                return pd.Timestamp(v)
            ts = pd.to_datetime(v, errors="coerce", dayfirst=True)
            if pd.isna(ts):
                d = parse_date(v, year)
                return pd.Timestamp(d) if d else pd.NaT
            return ts
        return pd.to_datetime(map_unique(values, conv), errors="coerce"), None
    if kind == "qty":
        parsed = map_unique(values, parse_qty)
        return pd.to_numeric(parsed.map(lambda x: x[0]), errors="coerce"), parsed.map(lambda x: x[1])
    return values.map(lambda v: None if v is None or (isinstance(v, float) and pd.isna(v)) else str(v).strip()), None


_ORDER_KEY_EXACT = {"hd", "ma", "so ct", "ct", "so chung tu", "ma ct", "so", "ma don", "don"}


def _order_key(df: pd.DataFrame, cols: dict) -> str | None:
    for c in df.columns:
        if c in LINEAGE or cols.get(c, {}).get("kind") in ("number", "date", "datetime", "empty"):
            continue
        f = fold(c)
        if f in _ORDER_KEY_EXACT or any(h in " " + f + " " for h in _ORDER_KEY_HINTS):
            return c
    return None


_SUMMARY_ROW = re.compile(r"^(tong cong|tong so|tong|cong)\b")


def _drop_summary_rows(df: pd.DataFrame, key: str | None) -> tuple[pd.DataFrame, int]:
    """Footer lines such as 'Tổng cộng' (no order key, a text cell starting with tổng/cộng)."""
    data_cols = [c for c in df.columns if c not in LINEAGE]
    def is_summary(row) -> bool:
        if key is not None and pd.notna(row.get(key)):
            return False
        return any(isinstance(v, str) and _SUMMARY_ROW.match(fold(v)) for v in row[data_cols].tolist())
    mask = df.apply(is_summary, axis=1) if len(df) else pd.Series([], dtype=bool)
    return df.loc[~mask], int(mask.sum())


def _ffill_order_level(df: pd.DataFrame, key: str, min_groups: int = 1) -> tuple[pd.DataFrame, list[str]]:
    """Orders are runs of lines with the same key (a blank key continues the order above). Forward-fill the key and
    every column that is constant inside multi-line orders and blank on some later line whose order's first line has
    it (rulebook §8). Needs `min_groups` multi-line orders to decide, so one stray line cannot trigger a fill."""
    data_cols = [c for c in df.columns if c not in LINEAGE]
    filled_key = df[key].ffill()
    grp = (filled_key != filled_key.shift()).cumsum()
    size = grp.map(grp.value_counts())
    multi = size > 1
    n_multi = int(grp[multi].nunique())
    out = df.copy()
    filled = []
    other = [c for c in data_cols if c != key]
    if df[key].isna().any() and df.loc[df[key].isna() & filled_key.notna(), other].notna().any(axis=None):
        out[key] = filled_key
        filled.append(key)
    if n_multi < min_groups:
        return out, filled
    is_first = ~grp.duplicated()
    for c in other:
        s = df[c]
        sub = pd.DataFrame({"g": grp[multi], "v": s[multi]}).dropna()
        if sub.empty:
            continue
        nuniq = sub.groupby("g")["v"].nunique()
        if (nuniq <= 1).mean() < 0.98:
            continue
        has_first = s.where(is_first).groupby(grp).transform("first").notna()
        gaps = s.isna() & ~is_first & has_first & multi
        if gaps.any():
            out[c] = s.groupby(grp).ffill()
            filled.append(c)
    return out, [c for c in df.columns if c in filled]


def clean_table(t: RawTable, readme_text: str) -> tuple[pd.DataFrame, dict]:
    df = t.df
    year = _year_hint(t.sheet, t.file)
    cols = {c: _infer_column(c, df[c], year) for c in df.columns if c not in LINEAGE}
    meta: dict = {"file": t.file, "sheet": t.sheet, "header_row": t.header_row, "rows_raw": int(len(df)), "year_hint": year}
    key = _order_key(df, cols)
    meta["order_key"] = key
    df, meta["summary_rows_dropped"] = _drop_summary_rows(df, key)
    work = df
    ffilled: list[str] = []
    if key is not None:
        work, ffilled = _ffill_order_level(df, key)
    meta["ffilled_columns"] = ffilled
    clean = pd.DataFrame(index=work.index)
    for c in work.columns:
        if c in LINEAGE:
            clean[c] = work[c]
            continue
        vals, unit = _clean_column(work[c], cols[c]["kind"], year)
        clean[c] = vals
        if unit is not None:
            clean[f"{c}__unit"] = unit
    dropped = 0
    if key is not None:
        data_cols = [c for c in clean.columns if c not in LINEAGE]
        dup = clean.duplicated(subset=data_cols, keep="first")
        dropped = int(dup.sum())
        clean = clean.loc[~dup]
    meta["duplicate_rows_dropped"] = dropped
    status_col = next((c for c in cols if "trang thai" in fold(c) or fold(c) in ("status", "tinh trang")), None)
    note_col = next((c for c in cols if "ghi chu" in fold(c) or fold(c) == "note"), None)
    if status_col is not None:
        clean["_status"] = map_unique(clean[status_col], norm_status)
        meta["status_from"] = status_col
    elif note_col is not None and key is not None:
        # rulebook §8: no status column -> status is in Ghi chú; a note naming no status is a completed order
        clean["_status"] = map_unique(clean[note_col], lambda v: norm_status(v) or "hoàn thành")
        meta["status_from"] = f"{note_col} (no status column)"
    if "_status" in clean:
        meta["status_counts"] = {str(k): int(v) for k, v in clean["_status"].value_counts(dropna=False).items()}
        unknown = clean.loc[clean["_status"].isna(), meta["status_from"].split(" (")[0]].dropna().astype(str).value_counts()
        meta["status_unmapped"] = {str(k): int(v) for k, v in unknown.head(10).items()}
    for c, info in cols.items():
        if info["kind"] in ("number",):
            s = clean[c]
            info.update(min=None if s.isna().all() else float(s.min()), max=None if s.isna().all() else float(s.max()))
        if info["kind"] in ("date", "datetime"):
            s = clean[c]
            info.update(min=None if s.isna().all() else str(s.min().date()), max=None if s.isna().all() else str(s.max().date()))
            info["parse_fail"] = int(s.isna().sum() - info["nulls"])
        examples = work[c].dropna().astype(str).head(3).tolist()
        info["examples"] = examples
    meta["columns"] = cols
    meta["rows_clean"] = int(len(clean))
    return clean.reset_index(drop=True), meta


def capability_profile(tables_meta: dict, readme_text: str) -> dict:
    all_cols = " | ".join(" " + fold(c) + " " for m in tables_meta.values() for c in m["columns"])
    readme = " " + fold(readme_text) + " "
    out = {}
    for cap, hints in _CAPABILITIES.items():
        if any(h in all_cols for h in hints):
            out[cap] = "present"
        elif "## cot" in readme or "cot trong" in readme:
            # the README lists the columns of every file and none matches: absent per README, still not proven
            out[cap] = "unknown" if any(h in readme for h in hints) else "absent_per_readme"
        else:
            out[cap] = "unknown"
    return out


_NOT_ACTIVITY_DATE = ("sinh", "han", "hsd", "expiry", "dang ky", "phai tra", "het han", "bao hanh")


def _date_range(tables_meta: dict) -> tuple[str | None, str | None]:
    """Span of activity dates: date columns of order tables (fallback: every table), minus birthdays, expiry and
    due dates, which lie far outside the data period."""
    lo, hi = None, None
    metas = [m for m in tables_meta.values() if m.get("order_key")] or list(tables_meta.values())
    for m in metas:
        for c, info in m["columns"].items():
            if any(re.search(rf"\b{h}\b", fold(c)) for h in _NOT_ACTIVITY_DATE):
                continue
            if info["kind"] in ("date", "datetime") and info.get("min"):
                lo = info["min"] if lo is None or info["min"] < lo else lo
                hi = info["max"] if hi is None or info["max"] > hi else hi
    return lo, hi


def shop_card(prof: dict, readme_text: str, max_cols: int = 40) -> str:
    """Compact text the helper code-agent reads: README + every clean table with typed columns and examples."""
    lines = [f"# Shop {prof['shop']}", "", readme_text.strip(), "", "## Bảng đã làm sạch (T['<tên>'] là pandas DataFrame)"]
    lines.append(f"Khoảng ngày thấy trong file: {prof['date_min']} .. {prof['date_max']}; README: dữ liệu đến hết {prof['data_until']}")
    for name, m in prof["tables"].items():
        src = m["file"] + (f" / sheet {m['sheet']}" if m["sheet"] else "")
        lines.append(f"\n### T['{name}']  ({src}; {m['rows_clean']} dòng)")
        notes = []
        if m.get("order_key"):
            notes.append(f"khoá đơn: {m['order_key']}")
        if m.get("ffilled_columns"):
            notes.append("đã ffill trong đơn: " + ", ".join(m["ffilled_columns"]))
        if m.get("duplicate_rows_dropped"):
            notes.append(f"đã bỏ {m['duplicate_rows_dropped']} dòng trùng cùng đơn")
        if m.get("status_from"):
            notes.append(f"_status (4 trạng thái chuẩn) lấy từ {m['status_from']}: {m.get('status_counts')}")
        if notes:
            lines.append("- " + "; ".join(notes))
        for i, (c, info) in enumerate(m["columns"].items()):
            if i >= max_cols:
                lines.append(f"- … {len(m['columns']) - max_cols} cột nữa")
                break
            extra = ""
            if info["kind"] in ("number", "date", "datetime") and info.get("min") is not None:
                extra = f" [{info['min']} .. {info['max']}]"
            elif info["kind"] == "text" and info.get("n_unique", 99) <= 12:
                extra = " giá trị: " + ", ".join(list(info.get("top", {}))[:12])
            elif info["kind"] == "qty":
                extra = f" (số; đơn vị ở cột '{c}__unit')"
            ex = "; ".join(info.get("examples", [])[:3])
            lines.append(f"- `{c}` {info['kind']}{extra} — vd: {ex}")
    lines.append("\n## Năng lực dữ liệu: " + ", ".join(f"{k}={v}" for k, v in prof["capabilities"].items()))
    return "\n".join(lines)


def cache_key(shop_dir: Path, extra: str = "") -> str:
    import hashlib

    h = hashlib.sha256(f"{LOADER_VERSION}/{PROFILER_VERSION}/{extra}".encode())
    for p in sorted(shop_dir.iterdir()):
        if p.is_file():
            h.update(p.name.encode())
            h.update(sha256_file(p).encode())
    from .provenance import digest
    code = {name: sha256_file(Path(__file__).with_name(name+'.py')) for name in ('loader','profiler','parsers')}
    return digest({'source':h.hexdigest(),'code':code})


def _to_parquet(df: pd.DataFrame, path: Path) -> None:
    df = df.copy()
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].map(lambda v: None if v is None or (isinstance(v, float) and pd.isna(v)) else str(v))
    df.to_parquet(path, index=False)


def profile_shop(shop_dir, out_root, split: str, rulebook_sha: str = "", force: bool = False) -> dict:
    """Profile one shop into out_root/<shop>/ (tables/*.parquet, raw/*.parquet, profile.json, card.md, COMPLETE)."""
    shop_dir = Path(shop_dir)
    out = Path(out_root) / shop_dir.name
    key = cache_key(shop_dir, f"{split}/{rulebook_sha}")
    marker = out / "COMPLETE"
    if not force and marker.exists() and marker.read_text().strip() == key:
        verify_shop_cache(shop_dir,out,cache_expectation(shop_dir,split,rulebook_sha))
        prof = json.loads((out / "profile.json").read_text(encoding="utf-8"))
        prof["reused"] = True
        return prof
    tmp = out.with_name(out.name + ".partial")
    shutil.rmtree(tmp, ignore_errors=True)
    (tmp / "tables").mkdir(parents=True)
    (tmp / "raw").mkdir()
    readme_path = shop_dir / "README.md"
    readme = readme_path.read_text(encoding="utf-8") if readme_path.exists() else ""
    tables_meta, errors = {}, {}
    for t in load_shop(shop_dir):
        try:
            clean, meta = clean_table(t, readme)
            _to_parquet(clean, tmp / "tables" / f"{t.name}.parquet")
            _to_parquet(t.df.astype(object), tmp / "raw" / f"{t.name}.parquet")
            tables_meta[t.name] = meta
        except Exception as e:  # one bad table must not hide the others
            errors[t.name] = f"{type(e).__name__}: {e}"
    rinfo = parse_readme(readme)
    lo, hi = _date_range(tables_meta)
    prof = {
        "shop": shop_dir.name,
        "split": split,
        "cache_key": key,
        "loader_version": LOADER_VERSION,
        "profiler_version": PROFILER_VERSION,
        "data_until": rinfo["data_until"],
        "conversions": rinfo["conversions"],
        "date_min": lo,
        "date_max": hi,
        "tables": tables_meta,
        "errors": errors,
        "capabilities": capability_profile(tables_meta, readme),
        "files": sorted(p.name for p in shop_dir.iterdir() if p.is_file()),
    }
    atomic_write_json(tmp / "profile.json", prof)
    atomic_write_text(tmp / "card.md", shop_card(prof, readme))
    expectation = cache_expectation(shop_dir,split,rulebook_sha)
    expectation['artifacts'] = {str(p.relative_to(tmp)):sha256_file(p) for p in sorted(tmp.rglob('*')) if p.is_file()}
    atomic_write_json(tmp/'cache_manifest.json',expectation)
    atomic_write_text(tmp / "COMPLETE", key)
    shutil.rmtree(out, ignore_errors=True)
    tmp.rename(out)
    return prof


def load_cached_shop(shop_cache_dir) -> tuple[dict[str, pd.DataFrame], dict]:
    d = Path(shop_cache_dir)
    if not (d / "COMPLETE").exists():
        raise FileNotFoundError(f"{d} has no COMPLETE marker")
    prof = json.loads((d / "profile.json").read_text(encoding="utf-8"))
    tables = {p.stem: pd.read_parquet(p) for p in sorted((d / "tables").glob("*.parquet"))}
    return tables, prof

def cache_expectation(shop_dir,split,rulebook_sha):
    from .provenance import tree_hash,digest
    code={name:sha256_file(Path(__file__).with_name(name+'.py')) for name in ('loader','profiler','parsers')}
    return {'schema':'dm-profile-v1','split':split,'source_sha256':tree_hash(shop_dir),
            'rulebook_sha256':rulebook_sha,'loader_code_sha256':digest(code)}


def verify_shop_cache(shop_dir,cache_dir,expected):
    from .provenance import tree_hash
    root=Path(cache_dir)
    if not (root/'COMPLETE').is_file() or not (root/'cache_manifest.json').is_file():raise ValueError(f'unverified profiler cache: {root}')
    got=json.loads((root/'cache_manifest.json').read_text())
    if tree_hash(shop_dir)!=expected['source_sha256']:raise ValueError('shop source changed')
    for key,value in expected.items():
        if got.get(key)!=value:raise ValueError(f'profiler cache mismatch: {key}')
    artifacts=got.get('artifacts',{})
    if not artifacts or not {'profile.json','card.md'}<=set(artifacts):raise ValueError('incomplete profiler artifact manifest')
    for name,sha in artifacts.items():
        p=(root/name).resolve()
        if not p.is_relative_to(root.resolve()) or not p.is_file() or sha256_file(p)!=sha:raise ValueError(f'profiler artifact changed: {name}')
    return got


In [ ]:
%%writefile /kaggle/working/lib/dmlib/rulebook.py
"""Split the split's rulebook.md into rules and pick the ones a question needs.

The rulebook changes between splits (new industries, new rules), so it is read at inference time. The general
sections (time, unanswerable, file conventions, answer format, words, revenue basics) always go into the prompt;
ranking / debt / stock rules are retrieved by word overlap with the question; an industry subsection is included
when the shop README mentions that industry.
"""
from __future__ import annotations

import math
import re
from collections import Counter
from dataclasses import dataclass

from .parsers import fold

ALWAYS = ("don hang va doanh thu", "thoi gian", "khong tra loi duoc", "cach ghi trong file", "dap an", "tu ngu")
INDUSTRY_ALIASES = {
    "ca phe": ("ca phe", "cafe", "coffee", "tra sua", "do uong"),
    "nha thuoc": ("nha thuoc", "thuoc", "duoc"),
    "vat lieu xay dung": ("vat lieu xay dung", "vlxd", "xi mang", "gach", "sat thep"),
    "me va be": ("me va be", "bim", "ta em be", "sua bot"),
    "sua xe": ("sua xe", "tiem xe", "xe may", "bien so"),
    "cua hang dien thoai": ("dien thoai", "iphone", "thu may cu", "tra gop"),
}


@dataclass
class Rule:
    section: str
    subsection: str | None
    title: str
    text: str  # the markdown row (or paragraph)
    header: str  # table header of the row, kept so the rule reads on its own


def _tokens(s: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", fold(s))


class Rulebook:
    def __init__(self, text: str):
        self.text = text
        self.preamble = ""
        self.sections: dict[str, str] = {}  # section title -> full markdown
        self.rules: list[Rule] = []
        self._parse()
        df = Counter(t for r in self.rules for t in set(_tokens(r.title + " " + r.text)))
        n = max(len(self.rules), 1)
        self.idf = {t: math.log(1 + n / c) for t, c in df.items()}

    def _parse(self) -> None:
        section = sub = None
        header = ""
        pre: list[str] = []
        buf: dict[str, list[str]] = {}
        lines = self.text.splitlines()
        for i, line in enumerate(lines):
            if line.startswith("## "):
                section, sub, header = line[3:].strip(), None, ""
                buf[section] = [line]
                continue
            if section is None:
                if not line.startswith("# "):
                    pre.append(line)
                continue
            buf[section].append(line)
            if line.startswith("### "):
                sub, header = line[4:].strip(), ""
                continue
            if line.startswith("|"):
                cells = [c.strip() for c in line.strip().strip("|").split("|")]
                nxt = lines[i + 1] if i + 1 < len(lines) else ""
                if re.fullmatch(r"\|?\s*-+.*", nxt) and set(nxt.replace("|", "").strip()) <= set("-: "):
                    header = line + "\n" + nxt
                    continue
                if set(line.replace("|", "").strip()) <= set("-: "):
                    continue
                self.rules.append(Rule(section, sub, re.sub(r"\*", "", cells[0]), line, header))
            elif line.strip().startswith("- "):
                self.rules.append(Rule(section, sub, line.strip()[2:60], line, ""))
        self.preamble = "\n".join(pre).strip()
        self.sections = {k: "\n".join(v).strip() for k, v in buf.items()}

    @staticmethod
    def _is_always(section: str) -> bool:
        f = fold(section)
        return any(a in f for a in ALWAYS)

    def _industries(self, industry_text: str) -> set[str]:
        f = " " + fold(industry_text) + " "
        out = set()
        for r in self.rules:
            if r.subsection:
                key = fold(r.subsection)
                aliases = INDUSTRY_ALIASES.get(key, (key,))
                if any(a in f for a in aliases):
                    out.add(r.subsection)
        return out

    def score(self, rule: Rule, question: str) -> float:
        q = set(_tokens(question))
        title = set(_tokens(rule.title))
        body = set(_tokens(rule.text))
        return sum(3 * self.idf.get(t, 0) for t in q & title) + sum(self.idf.get(t, 0) for t in q & (body - title))

    def select(self, question: str, industry_text: str = "", k: int = 8) -> str:
        """Markdown with the always-on sections, the shop's industry rules and the top-k other rules."""
        parts = [self.preamble] if self.preamble else []
        parts += [txt for title, txt in self.sections.items() if self._is_always(title)]
        industries = self._industries(industry_text)
        picked: dict[tuple, list[Rule]] = {}
        for r in self.rules:
            if self._is_always(r.section):
                continue
            if r.subsection and r.subsection in industries:
                picked.setdefault((r.section, r.subsection, r.header), []).append(r)
        others = [r for r in self.rules if not self._is_always(r.section) and not r.subsection]
        ranked = sorted(((self.score(r, question), j, r) for j, r in enumerate(others)), key=lambda x: (-x[0], x[1]))
        for s, _, r in ranked[:k]:
            if s > 0:
                picked.setdefault((r.section, None, r.header), []).append(r)
        for (section, sub, header), rules in picked.items():
            head = f"## {section}" + (f"\n### {sub}" if sub else "")
            parts.append(head + ("\n" + header if header else "") + "\n" + "\n".join(r.text for r in rules))
        return "\n\n".join(p for p in parts if p)

In [ ]:
%%writefile /kaggle/working/lib/dmlib/helpers.py
"""Stable arithmetic the helper code-agent calls (lane D). Business policy (what counts as revenue, debt, ties) stays
in the split's rulebook; these functions only implement the defaults that rulebook §1/§3/§9/§10 spell out."""
from __future__ import annotations

import pandas as pd

from .answer import round_half_up
from .parsers import fold
from .periods import in_range, pct_change  # noqa: F401  (re-exported for the agent)

_RANK = {"huỷ": 3, "trả hàng": 2, "đang giao": 1, "hoàn thành": 0}


def order_status(df: pd.DataFrame, order_col: str, status_col: str = "_status") -> pd.Series:
    """One status per order: a cancelled or (even partly) returned line marks the whole order."""
    st = df[status_col].fillna("hoàn thành").map(lambda s: _RANK.get(s, 0))
    worst = st.groupby(df[order_col]).max()
    inv = {v: k for k, v in _RANK.items()}
    return worst.map(inv)


def revenue_lines(df: pd.DataFrame, order_col: str, status_col: str = "_status") -> pd.DataFrame:
    """Lines of orders that count for revenue / order counts: not cancelled, not returned (whole order), đang giao kept."""
    st = order_status(df, order_col, status_col)
    ok = set(st[st.isin(["hoàn thành", "đang giao"])].index)
    return df[df[order_col].isin(ok)]


def rank(df: pd.DataFrame, value_col: str, name_col: str, code_col: str | None = None, k: int | None = None, ascending: bool = False) -> pd.DataFrame:
    """Sort by value (descending by default); equal values go by code ascending, else by name ascending."""
    tie = code_col or name_col
    out = df.sort_values([value_col, tie], ascending=[ascending, True], kind="mergesort")
    return out.head(k) if k else out


def base_factor(item: str, unit: str | None, conversions: dict) -> float:
    """How many base units one `unit` of `item` is (README / conversion table). Unknown unit or item -> 1."""
    if unit is None or (isinstance(unit, float) and pd.isna(unit)):
        return 1
    fi, fu = fold(item), fold(unit)
    for name, conv in conversions.items():
        if fold(name) != fi:
            continue
        if fold(conv.get("_base", "")) == fu:
            return 1
        for big, n in conv.items():
            if big != "_base" and fold(big) == fu:
                return n
    return 1


def money(x) -> int:
    """VND amount rounded to the đồng, half up."""
    return round_half_up(float(x), 0)


def pct(part, whole, ndigits: int = 2):
    """part / whole in percent, 2 decimals; None when whole is 0."""
    if not whole:
        return None
    return round_half_up(part / whole * 100, ndigits)

In [ ]:
%%writefile /kaggle/working/lib/dmlib/executor.py
"""Run generated code in a fresh Landlock child with resource and output limits."""
from __future__ import annotations
import json
import os
import re
import signal
import subprocess
import sys
import tempfile
import time
from dataclasses import dataclass,field
from pathlib import Path

LIB_ROOT=str(Path(__file__).resolve().parent.parent)
_BLOCKED=[(r"answers?\.jsonl|answers?\b['\"]",'label file'),
          (r'\b(import|from)\s+(requests|urllib|http|socket|aiohttp|httpx|ftplib|smtplib)\b','network'),
          (r'\b(import|from)\s+subprocess\b|os\.system|os\.popen|os\.exec|pty\.spawn','shell'),
          (r'shutil\.rmtree|os\.remove|os\.unlink|os\.rmdir|\.unlink\(','file deletion'),
          (r'/kaggle/input|\.\./','path outside the sandbox')]

@dataclass
class ExecResult:
    ok:bool
    record:dict|None=None
    error:str|None=None
    stdout:str=''
    seconds:float=0.
    extra:dict=field(default_factory=dict)


def static_check(code):
    for pattern,reason in _BLOCKED:
        if re.search(pattern,code):return f'blocked: {reason}'
    return None

_RUNNER=r'''
import json,sys,resource,traceback
from pathlib import Path
sys.path.insert(0,LIB_ROOT)
from dmlib.fs_guard import restrict_executor_files,runtime_read_paths
ctx=json.loads(Path('ctx.json').read_text())
code=Path('code.py').read_text()
preload=PRELOAD
policy=None
try:
    resource.setrlimit(resource.RLIMIT_AS,(MEM_MB<<20,MEM_MB<<20))
    resource.setrlimit(resource.RLIMIT_CPU,(CPU_S,CPU_S+1))
    resource.setrlimit(resource.RLIMIT_FSIZE,(2<<20,2<<20))
    reads=runtime_read_paths(MODE)+[ctx['DATA_DIR'],ctx['CALENDAR_PATH']]
    if MODE=='helper':reads.append(ctx['SHOP_CACHE'])
    policy=restrict_executor_files(reads,str(Path.cwd()))
    from dmlib.answer import format_answer
    _done=[]
    def _write(rec):
        _done.append(rec)
        Path('result.json').write_text(json.dumps({'ok':True,'record':rec,'policy':policy},ensure_ascii=False,allow_nan=False))
    def submit(type,value=None,reason=None,money=False):
        _write(format_answer(ctx['QID'],type,value,reason=reason,money=money))
    def unanswerable(reason='dữ liệu không đủ để trả lời'):
        submit('unanswerable',reason=str(reason))
    def abstain(reason=None):submit('abstain')
    globals().update(ctx)
    exec(compile(preload,'<preload>','exec'),globals())
    exec(compile(code,'<agent_code>','exec'),globals())
    if len(_done)!=1:raise RuntimeError('code must submit exactly one final answer')
except BaseException:
    Path('result.json').write_text(json.dumps({'ok':False,'error':traceback.format_exc()[-4000:],'policy':policy}))
'''


def run_code(code,qid,context,timeout_s=60,mem_mb=6144,max_output_chars=4000,preload='',workdir=None,python=None):
    why=static_check(code)
    if why:return ExecResult(False,error=why)
    started=time.time()
    # Dedicated interpreter: the base environment supplies pandas/calamine/parquet; vLLM stays isolated.
    interpreter=python or os.environ.get('DM_EXECUTOR_PYTHON') or sys.executable
    with tempfile.TemporaryDirectory(prefix='dm_exec_',dir=workdir) as name:
        root=Path(name);mode='helper' if 'SHOP_CACHE' in context else 'raw'
        (root/'ctx.json').write_text(json.dumps({**context,'QID':qid},ensure_ascii=False))
        (root/'code.py').write_text(code)
        prefix=f'LIB_ROOT={LIB_ROOT!r}\nPRELOAD={preload!r}\nMEM_MB={int(mem_mb)}\nCPU_S={max(1,int(timeout_s)+1)}\nMODE={mode!r}\n'
        (root/'runner.py').write_text(prefix+_RUNNER)
        env={'PATH':os.environ.get('PATH','/usr/bin:/bin'),'HOME':name,'TMPDIR':name,
             'PYTHONDONTWRITEBYTECODE':'1','PYTHONIOENCODING':'utf-8','OMP_NUM_THREADS':'1',
             'MKL_NUM_THREADS':'1','OPENBLAS_NUM_THREADS':'1','MPLBACKEND':'Agg'}
        with open(root/'stdout.log','wb') as log:
            proc=subprocess.Popen([interpreter,str(root/'runner.py')],cwd=root,env=env,stdin=subprocess.DEVNULL,
                                  stdout=log,stderr=subprocess.STDOUT,close_fds=True,start_new_session=True)
            try:proc.wait(timeout=timeout_s)
            except subprocess.TimeoutExpired:
                os.killpg(proc.pid,signal.SIGKILL);proc.wait()
                return ExecResult(False,error=f'timeout after {timeout_s}s',seconds=time.time()-started)
        with open(root/'stdout.log','rb') as log:stdout=log.read(max_output_chars).decode('utf-8','replace')
        result=root/'result.json'
        if not result.is_file():return ExecResult(False,error=f'process exited {proc.returncode} without result',stdout=stdout,seconds=time.time()-started)
        try:res=json.loads(result.read_text())
        except (ValueError,OSError) as e:return ExecResult(False,error=f'invalid executor result: {e}',stdout=stdout,seconds=time.time()-started)
        return ExecResult(bool(res.get('ok') and proc.returncode==0),record=res.get('record'),error=res.get('error'),
                          stdout=stdout,seconds=time.time()-started,extra={'policy':res.get('policy')})


In [ ]:
%%writefile /kaggle/working/lib/dmlib/llm.py
"""Greedy chat generation behind one interface: vLLM (fast, Kaggle T4 in fp16), HF transformers (fallback) or a mock.

Prompts are rendered with the model's own chat template (enable_thinking passed when the template knows it) and
prompts that do not fit `max_model_len - max_tokens` are not generated but returned with finish='too_long'.
"""
from __future__ import annotations

import os
import random
from dataclasses import dataclass
from typing import Callable


@dataclass
class GenOut:
    text: str
    prompt_tokens: int
    output_tokens: int
    finish: str  # stop | length | too_long | error


def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import numpy as np

        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


class BaseLLM:
    max_model_len: int = 16384

    def render(self, conv: list[dict]) -> str:
        raise NotImplementedError

    def count_tokens(self, conv: list[dict]) -> int:
        raise NotImplementedError

    def generate(self, convs: list[list[dict]], max_tokens: int) -> list[GenOut]:
        raise NotImplementedError


def _apply_template(tok, conv, thinking: bool) -> str:
    try:
        return tok.apply_chat_template(conv, tokenize=False, add_generation_prompt=True, enable_thinking=thinking)
    except TypeError:
        return tok.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)


class VLLM(BaseLLM):
    def __init__(self, model: str, revision: str | None = None, dtype: str = "half", max_model_len: int = 16384,
                 gpu_memory_utilization: float = 0.90, quantization: str | None = None, seed: int = 0,
                 enable_thinking: bool = False, tensor_parallel_size: int = 1, enforce_eager: bool = False,
                 max_num_seqs: int | None = None, max_num_batched_tokens: int | None = None, enable_chunked_prefill: bool | None = None, **_):
        from vllm import LLM, SamplingParams  # noqa: F401

        self._SamplingParams = SamplingParams
        scheduling = {k:v for k,v in {"max_num_seqs":max_num_seqs,"max_num_batched_tokens":max_num_batched_tokens,
                      "enable_chunked_prefill":enable_chunked_prefill}.items() if v is not None}
        self.llm = LLM(model=model, revision=revision, dtype=dtype, max_model_len=max_model_len, seed=seed,
                       gpu_memory_utilization=gpu_memory_utilization, quantization=quantization,
                       tensor_parallel_size=tensor_parallel_size, enable_prefix_caching=True, enforce_eager=enforce_eager,
                       **scheduling)
        self.tok = self.llm.get_tokenizer()
        self.max_model_len = max_model_len
        self.thinking = enable_thinking

    def render(self, conv):
        return _apply_template(self.tok, conv, self.thinking)

    def count_tokens(self, conv):
        return len(self.tok(self.render(conv), add_special_tokens=False)["input_ids"])

    def generate(self, convs, max_tokens):
        prompts = [self.render(c) for c in convs]
        lens = [len(self.tok(p, add_special_tokens=False)["input_ids"]) for p in prompts]
        todo = [i for i, n in enumerate(lens) if n + max_tokens <= self.max_model_len]
        outs = [GenOut("", n, 0, "too_long") for n in lens]
        if todo:
            sp = self._SamplingParams(temperature=0.0, top_p=1.0, max_tokens=max_tokens, seed=0)
            res = self.llm.generate([prompts[i] for i in todo], sp, use_tqdm=False)
            for i, r in zip(todo, res):
                o = r.outputs[0]
                outs[i] = GenOut(o.text, lens[i], len(o.token_ids), "length" if o.finish_reason == "length" else "stop")
        return outs


class HFLLM(BaseLLM):
    def __init__(self, model: str, revision: str | None = None, dtype: str = "half", max_model_len: int = 16384,
                 load_in_4bit: bool = False, enable_thinking: bool = False, batch_size: int = 4, device_map="auto", **_):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.torch = torch
        self.tok = AutoTokenizer.from_pretrained(model, revision=revision)
        self.tok.padding_side = "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        kw = {"revision": revision, "device_map": device_map}
        if load_in_4bit:
            from transformers import BitsAndBytesConfig

            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
        else:
            kw["torch_dtype"] = torch.float16 if dtype in ("half", "float16") else (torch.bfloat16 if dtype == "bfloat16" else torch.float32)
        self.model = AutoModelForCausalLM.from_pretrained(model, **kw).eval()
        self.max_model_len = max_model_len
        self.thinking = enable_thinking
        self.batch_size = batch_size

    def render(self, conv):
        return _apply_template(self.tok, conv, self.thinking)

    def count_tokens(self, conv):
        return len(self.tok(self.render(conv), add_special_tokens=False)["input_ids"])

    def generate(self, convs, max_tokens):
        prompts = [self.render(c) for c in convs]
        lens = [len(self.tok(p, add_special_tokens=False)["input_ids"]) for p in prompts]
        outs = [GenOut("", n, 0, "too_long") for n in lens]
        todo = [i for i, n in enumerate(lens) if n + max_tokens <= self.max_model_len]
        todo.sort(key=lambda i: lens[i])
        for b in range(0, len(todo), self.batch_size):
            idx = todo[b : b + self.batch_size]
            enc = self.tok([prompts[i] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False).to(self.model.device)
            with self.torch.no_grad():
                gen = self.model.generate(**enc, max_new_tokens=max_tokens, do_sample=False, temperature=None, top_p=None,
                                          top_k=None, pad_token_id=self.tok.pad_token_id)
            for row, i in zip(gen, idx):
                new = row[enc["input_ids"].shape[1] :]
                n_out = int((new != self.tok.pad_token_id).sum())
                text = self.tok.decode(new, skip_special_tokens=True)
                outs[i] = GenOut(text, lens[i], n_out, "length" if n_out >= max_tokens else "stop")
        return outs


class MockLLM(BaseLLM):
    """Deterministic stand-in for tests and CPU dry-runs: fn(conv) -> text."""

    def __init__(self, fn: Callable[[list[dict]], str], max_model_len: int = 16384, **_):
        self.fn = fn
        self.max_model_len = max_model_len

    def render(self, conv):
        return "\n".join(f"<{m['role']}>{m['content']}" for m in conv)

    def count_tokens(self, conv):
        return len(self.render(conv)) // 3

    def generate(self, convs, max_tokens):
        out = []
        for c in convs:
            n = self.count_tokens(c)
            out.append(GenOut("", n, 0, "too_long") if n + max_tokens > self.max_model_len else GenOut(self.fn(c), n, 1, "stop"))
        return out


def make_llm(cfg: dict, mock_fn: Callable | None = None) -> BaseLLM:
    """cfg: backend (auto|vllm|hf|mock), model, revision, dtype, max_model_len, gpu_memory_utilization, quantization,
    load_in_4bit, enable_thinking, seed."""
    backend = cfg.get("backend", "auto")
    seed_everything(int(cfg.get("seed", 0)))
    if backend == "mock":
        return MockLLM(mock_fn or (lambda conv: "{\"done\": true}"), **cfg)
    if backend in ("auto", "vllm"):
        try:
            return VLLM(**{k: v for k, v in cfg.items() if k != "backend"})
        except Exception as e:  # not installed, or no kernel for this GPU (e.g. T4): fall back to transformers
            if backend == "vllm":
                raise
            print(f"[llm] vLLM unavailable ({type(e).__name__}: {e}); using transformers", flush=True)
    return HFLLM(**{k: v for k, v in cfg.items() if k != "backend"})

In [ ]:
%%writefile /kaggle/working/lib/dmlib/agent.py
"""DataMate code-agent: one generated solution per question, at most one repair after an execution error, abstain
when the final attempt still fails. Two architectures share everything except the shop context:

  raw    (lane C): README + the first raw rows of every file/sheet; the code reads the files itself with pandas.
  helper (lane D): profiler card of cleaned tables T[...] + helper library (periods P, helpers H, conversions CONV).
"""
from __future__ import annotations

import re
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd

from .executor import run_code
from .loader import read_csv_grid, read_excel_grids
from .rulebook import Rulebook

SYSTEM_COMMON = """Bạn là chuyên gia phân tích dữ liệu bán hàng của hộ kinh doanh Việt Nam. Hãy viết MỘT khối ```python``` duy nhất tính đáp án cho câu hỏi.

Quy tắc bắt buộc:
- Hôm nay là biến AS_OF (chuỗi 'YYYY-MM-DD'). Bản ghi có ngày sau hôm nay coi như chưa xảy ra (riêng trạng thái huỷ/trả hàng ghi trên đơn vẫn lấy).
- Áp dụng đúng RULEBOOK bên dưới; câu hỏi nói khác thì theo câu hỏi.
- Kết thúc code bằng đúng một lệnh:
  submit(type, value)  với type là number | percent | string | date | boolean | list | table
  unanswerable("lý do")  khi shop không ghi loại thông tin cần, hoặc kỳ hỏi nằm HOÀN TOÀN ngoài khoảng ngày có dữ liệu (kỳ chỉ chồng một phần thì vẫn tính trên phần có dữ liệu)
  abstain()  khi không đủ chắc chắn (sai bị trừ điểm, bỏ trống được 0 điểm).
- number: tiền là số nguyên VND (dùng submit('number', x, money=True) để làm tròn về đồng); số đếm là số nguyên; số khác làm tròn 2 chữ số.
- percent: là số phần trăm (12,4% -> 12.4), giảm thì âm, 2 chữ số thập phân.
- string: tên đúng như ghi trong dữ liệu (hoặc mã). date: 'YYYY-MM-DD' (câu hỏi theo tháng: 'YYYY-MM'). boolean: True/False.
- list: list Python có thứ tự (một phần tử vẫn là list); câu hỏi nhiều số thì theo đúng thứ tự câu hỏi nêu.
- table: {"columns": [tên cột...], "rows": [[...], ...]}; cột đầu là tên đối tượng, các cột sau là số đo theo thứ tự câu hỏi.
- Bằng nhau thì xếp theo mã tăng dần (không có mã thì theo tên).
- print() các số trung gian quan trọng (số dòng sau lọc, kỳ thời gian, tổng) để kiểm tra.
- Chỉ đọc file trong DATA_DIR và CALENDAR_PATH (lịch âm: cột date, lunar_day, lunar_month, lunar_year, lunar_leap). Không dùng mạng, không ghi/xoá file dữ liệu.
"""

SYSTEM_RAW = SYSTEM_COMMON + """
Dữ liệu là file THÔ trong thư mục DATA_DIR (xlsx/csv/sqlite). Bạn tự đọc bằng pandas (pd.read_excel(..., sheet_name=None, header=None), pd.read_csv, sqlite3), tự tìm dòng tiêu đề, tự xử lý: ô trống ở các dòng sau của cùng một đơn kế thừa dòng đầu; dòng trùng y hệt trong cùng một đơn chỉ tính một lần; tiền dạng '144.000đ', '750k', '1tr200', '1,45tr', '10,000.0000'; ngày dạng '1 thg 2 2025', '01/02/2025', ngày không ghi năm lấy năm của sheet; trạng thái chuẩn hoá về hoàn thành / huỷ / trả hàng / đang giao.
Biến có sẵn: DATA_DIR, CALENDAR_PATH, AS_OF, pd, np.
"""

SYSTEM_HELPER = SYSTEM_COMMON + """
Dữ liệu đã được đọc và làm sạch sẵn: T là dict tên bảng -> pandas DataFrame (mô tả bảng ở phần SHOP).
- Cột tiền/số đã là số; cột ngày là datetime64; cột số lượng kèm đơn vị có thêm cột '<tên>__unit'.
- Ô trống trong đơn đã được ffill (xem 'đã ffill trong đơn'); dòng trùng cùng đơn đã bỏ; cột _status là trạng thái chuẩn (hoàn thành / huỷ / trả hàng / đang giao) nếu có.
- Cột _file, _sheet, _row cho biết nguồn của dòng.
Thư viện có sẵn:
- P (dmlib.periods): P.this_week(AS_OF), P.last_week(AS_OF), P.last_n_days(AS_OF, n), P.this_month(AS_OF), P.same_days_prev_month(AS_OF), P.prev_month_full(AS_OF), P.month(m, year=None, as_of=AS_OF), P.quarter(q, year=None, as_of=AS_OF), P.year_to_date(AS_OF), P.day(d), P.yesterday(AS_OF), P.prev_period(p), P.clip_to_asof(p, AS_OF), P.tet_range(year, CAL), P.before_tet_range(year, CAL), P.shift_of(ts) -> 'sáng'|'chiều'|None, P.count_weekday(p, weekday_monday0), P.in_range(series, p) -> mask, P.overlap_status(p, (start, end)) -> 'outside'|'partial'|'inside'. Mỗi kỳ là tuple (date_bắt_đầu, date_kết_thúc) gồm cả hai đầu.
- H (dmlib.helpers): H.order_status(df, order_col) -> trạng thái mỗi đơn (dòng trả/huỷ đánh dấu cả đơn); H.revenue_lines(df, order_col) -> các dòng của đơn tính doanh thu; H.rank(df, value_col, name_col, code_col=None, k=None, ascending=False); H.base_factor(item, unit, CONV) -> hệ số quy về đơn vị cơ bản; H.money(x); H.pct(part, whole); H.pct_change(cur, prev).
- CONV: quy đổi đơn vị trong README {tên hàng: {đơn vị lớn: hệ số, '_base': đơn vị cơ bản}}; CAL: DataFrame lịch âm; PROFILE: hồ sơ shop (data_until, date_min, date_max, capabilities).
Biến có sẵn: T, P, H, CONV, CAL, PROFILE, AS_OF, DATA_DIR, CALENDAR_PATH, pd, np.
"""

PRELOAD_RAW = "import pandas as pd, numpy as np"
PRELOAD_HELPER = """import pandas as pd, numpy as np
from dmlib.profiler import load_cached_shop
from dmlib import helpers as H, periods as P
T, PROFILE = load_cached_shop(SHOP_CACHE)
CONV = PROFILE.get("conversions", {})
CAL = pd.read_csv(CALENDAR_PATH)"""

_CODE_RE = re.compile(r"```(?:python|py)?\s*\n(.*?)```", re.S)


def extract_code(text: str) -> str | None:
    """The last fenced block that ends the answer (submit/unanswerable/abstain), else the longest block."""
    blocks = [b.strip() for b in _CODE_RE.findall(text or "")]
    if blocks:
        final = [b for b in blocks if re.search(r"\b(submit|unanswerable|abstain)\(", b)]
        return final[-1] if final else max(blocks, key=len)
    t = (text or "").strip()
    if t.startswith("```"):
        t = t.strip("`").split("\n", 1)[-1]
    return t if re.search(r"\b(submit|unanswerable|abstain)\(", t) else None


def _cell(v, width=40) -> str:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    s = str(v).replace("\n", " ")
    return s if len(s) <= width else s[: width - 1] + "…"


def raw_shop_context(shop_dir: Path, head_rows: int = 8, max_chars: int = 9000) -> str:
    """README + the first raw rows (no header guessing) of every sheet / csv / sqlite table."""
    import sqlite3

    readme = (shop_dir / "README.md").read_text(encoding="utf-8") if (shop_dir / "README.md").exists() else ""
    parts = [readme.strip(), "", "## Các file (vài dòng đầu, chưa xử lý)"]
    for p in sorted(shop_dir.iterdir()):
        suf = p.suffix.lower()
        grids = {}
        if suf in (".xlsx", ".xlsm", ".xls"):
            for sh, g in read_excel_grids(p).items():
                grids[f"{p.name} | sheet '{sh}'"] = g
        elif suf in (".csv", ".tsv", ".txt"):
            grids[p.name] = read_csv_grid(p)
        elif suf in (".sqlite", ".db", ".sqlite3"):
            con = sqlite3.connect(f"file:{p}?mode=ro", uri=True)
            for (t,) in con.execute("select name from sqlite_master where type='table'"):
                n = con.execute(f'select count(*) from "{t}"').fetchone()[0]
                g = pd.read_sql_query(f'select * from "{t}" limit {head_rows}', con)
                g = pd.concat([pd.DataFrame([g.columns.tolist()]), pd.DataFrame(g.values)], ignore_index=True)
                grids[f"{p.name} | bảng sqlite '{t}' ({n} dòng)"] = g
            con.close()
        else:
            continue
        for title, g in grids.items():
            parts.append(f"\n### {title}  ({len(g)} dòng thô, {g.shape[1]} cột)")
            for _, row in g.head(head_rows).iterrows():
                parts.append(" | ".join(_cell(v) for v in row.tolist()))
    text = "\n".join(parts)
    return text if len(text) <= max_chars else text[:max_chars] + "\n…[cắt bớt]"


def helper_shop_context(shop_cache: Path, max_chars: int = 9000) -> str:
    text = (shop_cache / "card.md").read_text(encoding="utf-8")
    return text if len(text) <= max_chars else text[:max_chars] + "\n…[cắt bớt]"


@dataclass
class AgentConfig:
    mode: str = "helper"  # raw | helper
    max_new_tokens: int = 1536
    exec_timeout_s: float = 60
    exec_mem_mb: int = 6144
    exec_workers: int = 1
    repair: bool = True
    rules_k: int = 8
    context_chars: int = 9000


@dataclass
class Item:
    id: str
    business_id: str
    question: str
    as_of_date: str


@dataclass
class Outcome:
    record: dict
    status: str  # completed | retryable
    trace: dict = field(default_factory=dict)


class ShopContexts:
    """Per-shop prompt context and executor variables, computed once per shop."""

    def __init__(self, split_dir: Path, mode: str, cache_dir: Path | None, context_chars: int):
        self.split_dir, self.mode, self.cache_dir, self.context_chars = Path(split_dir), mode, cache_dir, context_chars
        self._ctx: dict[str, str] = {}

    def shop_dir(self, shop: str) -> Path:
        return self.split_dir / "data" / shop

    def text(self, shop: str) -> str:
        if shop not in self._ctx:
            if self.mode == "raw":
                self._ctx[shop] = raw_shop_context(self.shop_dir(shop), max_chars=self.context_chars)
            else:
                self._ctx[shop] = helper_shop_context(Path(self.cache_dir) / shop, max_chars=self.context_chars)
        return self._ctx[shop]

    def exec_context(self, item: Item) -> dict:
        ctx = {"DATA_DIR": str(self.shop_dir(item.business_id)), "CALENDAR_PATH": str(self.split_dir / "calendar.csv"), "AS_OF": item.as_of_date}
        if self.mode == "helper":
            ctx["SHOP_CACHE"] = str(Path(self.cache_dir) / item.business_id)
        return ctx

    def industry_text(self, shop: str) -> str:
        p = self.shop_dir(shop) / "README.md"
        return p.read_text(encoding="utf-8")[:1500] if p.exists() else ""


def build_messages(item: Item, shops: ShopContexts, rulebook: Rulebook, cfg: AgentConfig) -> list[dict]:
    system = SYSTEM_RAW if cfg.mode == "raw" else SYSTEM_HELPER
    rules = rulebook.select(item.question, industry_text=shops.industry_text(item.business_id), k=cfg.rules_k)
    user = (
        f"# RULEBOOK (trích)\n{rules}\n\n# SHOP {item.business_id}\n{shops.text(item.business_id)}\n\n"
        f"# CÂU HỎI\nHôm nay (AS_OF): {item.as_of_date}\nCâu hỏi: {item.question}\n\nViết code Python:"
    )
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def repair_messages(conv: list[dict], reply: str, error: str, stdout: str) -> list[dict]:
    msg = f"Code chạy lỗi:\n{(error or '')[-2500:]}\n"
    if stdout:
        msg += f"\nstdout:\n{stdout[-1500:]}\n"
    msg += "\nHãy sửa và trả về một khối ```python``` hoàn chỉnh (vẫn kết thúc bằng submit / unanswerable / abstain)."
    return conv + [{"role": "assistant", "content": reply}, {"role": "user", "content": msg}]


def _abstain(qid: str) -> dict:
    return {"id": qid, "type": "abstain"}


def solve_batch(items, llm, rulebook, shops, cfg, on_progress=None, resume_states=None):
    """Coordinator emits durable stages; backend failures never become a completed abstain.

    resume_states contains the latest event for each ID. first_generated reuses its reply;
    repair_pending reuses the first attempt and spends only the outstanding repair.
    """
    from concurrent.futures import as_completed
    from dataclasses import asdict
    from .answer import validate_record
    resume_states=resume_states or {};outcomes={};states={};convs={};need_first=[];need_exec=[];repair=[]
    preload=PRELOAD_RAW if cfg.mode=='raw' else PRELOAD_HELPER
    def emit(i,stage,trace,record=None,generation=None):
        event={'id':items[i].id,'stage':stage,'status':'completed' if stage=='completed' else 'retryable',
               'trace':trace, 'record':record, 'generation':generation}
        states[i]=event
        if on_progress:on_progress(event)
        return event
    def finish(i,trace,record=None,reason=None):
        record=record or _abstain(items[i].id)
        trace.update(stage='completed',final=record,abstain_reason=reason)
        outcomes[i]=Outcome(record,'completed',trace)
        emit(i,'completed',trace,record=record)
    def pending(i,trace,stage,error,generation=None):
        trace.update(stage=stage,infrastructure_error=str(error))
        outcomes[i]=Outcome(_abstain(items[i].id),'retryable',trace)
        emit(i,stage,trace,generation=generation)
    for i,it in enumerate(items):
        old=resume_states.get(it.id,{})
        tr=old.get('trace') or {'id':it.id,'shop':it.business_id,'question':it.question,'as_of':it.as_of_date,'mode':cfg.mode,'attempts':[]}
        try:convs[i]=build_messages(it,shops,rulebook,cfg)
        except Exception as e:
            tr['loader_error']=f'{type(e).__name__}: {e}';finish(i,tr,reason='loader: '+tr['loader_error']);continue
        stage=old.get('stage')
        if stage=='repair_pending':
            states[i]=old
            if old.get('generation'):need_exec.append((i,old['generation'],True))
            else:repair.append(i)
        elif stage=='first_generated' and old.get('generation'):
            states[i]=old;need_exec.append((i,old['generation'],False))
        else:need_first.append(i);states[i]={'trace':tr}
    if need_first:
        try:
            gens=llm.generate([convs[i] for i in need_first],cfg.max_new_tokens)
            if len(gens)!=len(need_first):raise RuntimeError('generation cardinality mismatch')
        except Exception as e:
            for i in need_first:pending(i,states[i]['trace'],'pending',e)
            gens=[]
        for i,g in zip(need_first,gens):
            if g.finish=='error':pending(i,states[i]['trace'],'pending','generation error');continue
            gen=asdict(g);emit(i,'first_generated',states[i]['trace'],generation=gen);need_exec.append((i,gen,False))
    def execute(i,gen,is_repair):
        code=extract_code(gen['text']);res=None
        if code is not None and gen['finish']!='too_long':
            res=run_code(code,items[i].id,shops.exec_context(items[i]),timeout_s=cfg.exec_timeout_s,mem_mb=cfg.exec_mem_mb,preload=preload)
        attempt={'reply':gen['text'],'code':code,'prompt_tokens':gen['prompt_tokens'],'output_tokens':gen['output_tokens'],
                 'finish':gen['finish'],'ok':bool(res and res.ok),'error':res.error if res else ('prompt too long' if gen['finish']=='too_long' else 'no code block in reply'),
                 'stdout':res.stdout if res else '', 'exec_s':res.seconds if res else 0.,'policy':res.extra.get('policy') if res else None}
        if res and res.ok:
            error=validate_record(res.record)
            if error:attempt.update(ok=False,error='format: '+error)
        return i,gen,is_repair,attempt,(res.record if res and attempt['ok'] else None)
    def execute_wave(tasks):
        with ThreadPoolExecutor(max_workers=max(1,cfg.exec_workers)) as pool:
            futures=[pool.submit(execute,*task) for task in tasks]
            for f in as_completed(futures):
                i,gen,is_repair,attempt,record=f.result();tr=states[i]['trace']
                tr['attempts'].append(attempt)
                if record is not None:finish(i,tr,record)
                elif not is_repair and cfg.repair and gen['finish']!='too_long':
                    emit(i,'repair_pending',tr);repair.append(i)
                else:finish(i,tr,reason=attempt['error'] or 'failed after repair')
    execute_wave(need_exec)
    if repair:
        repair=list(dict.fromkeys(repair))
        prompts=[]
        for i in repair:
            a=states[i]['trace']['attempts'][0]
            prompts.append(repair_messages(convs[i],a['reply'],a['error'],a.get('stdout','')))
        try:
            gens=llm.generate(prompts,cfg.max_new_tokens)
            if len(gens)!=len(repair):raise RuntimeError('repair cardinality mismatch')
        except Exception as e:
            for i in repair:pending(i,states[i]['trace'],'repair_pending',e)
            gens=[]
        tasks=[]
        for i,g in zip(repair,gens):
            if g.finish=='error':pending(i,states[i]['trace'],'repair_pending','repair backend error')
            else:
                gen=asdict(g);emit(i,'repair_pending',states[i]['trace'],generation=gen)
                tasks.append((i,gen,True))
        execute_wave(tasks)
    return [outcomes[i] for i in range(len(items))]


In [ ]:
%%writefile /kaggle/working/lib/dmlib/split.py
"""Fixed shop-level manifest for train: train-fit / dev-tune / final-holdout (no shop in two parts)."""
from __future__ import annotations

import random
from collections import Counter


def make_manifest(questions: list[dict], seed: int = 2026, dev_shops: int = 6, final_shops: int = 6) -> dict:
    shops = sorted({q["business_id"] for q in questions})
    rng = random.Random(seed)
    rng.shuffle(shops)
    final, dev, fit = shops[:final_shops], shops[final_shops : final_shops + dev_shops], shops[final_shops + dev_shops :]
    part = {s: "final_holdout" for s in final} | {s: "dev_tune" for s in dev} | {s: "train_fit" for s in fit}
    ids = {"train_fit": [], "dev_tune": [], "final_holdout": []}
    for q in questions:
        ids[part[q["business_id"]]].append(q["id"])
    return {
        "seed": seed,
        "shops": {"train_fit": sorted(fit), "dev_tune": sorted(dev), "final_holdout": sorted(final)},
        "ids": ids,
        "counts": {k: len(v) for k, v in ids.items()},
        "questions_per_shop": dict(Counter(q["business_id"] for q in questions)),
    }

def prepare_dev_run(cfg: dict, questions: list[dict]) -> dict:
    from .provenance import digest, package_hash
    from pathlib import Path
    ids=[q['id'] for q in questions]
    if len(ids)!=len(set(ids)):raise ValueError('duplicate question IDs')
    clean=[{k:q[k] for k in ('id','business_id','question','as_of_date')} for q in sorted(questions,key=lambda q:q['id'])]
    manifest=make_manifest(clean,seed=cfg.get('manifest_seed',2026))
    subset=cfg.get('subset','dev_tune')
    selected=sorted(set(subset) if isinstance(subset,list) else (set(ids) if subset=='all' else set().union(*(manifest['ids'][k] for k in subset.split('+')))))
    if not selected:raise ValueError('selected dev manifest is empty')
    if not set(selected)<=set(ids):raise ValueError('unknown question ID')
    manifest.update(selected_ids=selected,input_sha256=digest(clean),code_sha256=package_hash(Path(__file__).parent))
    manifest['manifest_sha256']=digest({'ids':manifest['ids'],'selected_ids':selected,'shops':manifest['shops'],'seed':manifest['seed']})
    grouped={shop:sorted(q['id'] for q in clean if q['business_id']==shop) for shop in manifest['shops']['train_fit']}
    smoke=[]
    while grouped and len(smoke)<12:
        for shop in sorted(list(grouped)):
            if grouped[shop]:smoke.append(grouped[shop].pop(0))
            if not grouped[shop]:del grouped[shop]
            if len(smoke)==12:break
    manifest['smoke_ids']=sorted(smoke)
    return manifest


In [ ]:
%%writefile /kaggle/working/lib/dmlib/run.py
"""One stable ID shard per GPU; coordinator checkpoints every generated/executed stage."""
from __future__ import annotations
import argparse
import json
import os
import subprocess
import sys
import time
from pathlib import Path
from .agent import AgentConfig,Item,ShopContexts,solve_batch
from .common import DurableJsonl,atomic_write_json,env_info,find_split_dir,load_jsonl,set_thread_env,shard_of,session_deadline,can_start,MemoryMonitor
from .provenance import semantic_hash
from .rulebook import Rulebook
from .split import make_manifest


def resolve_split_dir(cfg):
    return Path(cfg['split_dir']) if cfg.get('split_dir') else find_split_dir(cfg['split'],cfg.get('input_roots',['/kaggle/input']),marker='questions.jsonl')


def select_items(cfg,split_dir):
    qs=load_jsonl(Path(split_dir)/'questions.jsonl');subset=cfg.get('subset','all')
    if subset!='all':
        manifest=make_manifest(qs,seed=cfg.get('manifest_seed',2026))
        keep=set(subset) if isinstance(subset,list) else set().union(*(manifest['ids'][part] for part in subset.split('+')))
        qs=[q for q in qs if q['id'] in keep]
    if cfg.get('shops'):qs=[q for q in qs if q['business_id'] in set(cfg['shops'])]
    if cfg.get('limit'):qs=qs[:int(cfg['limit'])]
    return [Item(q['id'],q['business_id'],q['question'],q['as_of_date']) for q in qs]


def run_hash(cfg):return semantic_hash(cfg)


def work(cfg,shard_index,llm=None):
    set_thread_env(int(cfg.get('cpu_threads',1)));deadline=session_deadline(cfg)
    split_dir=resolve_split_dir(cfg);out=Path(cfg['out_dir']);out.mkdir(parents=True,exist_ok=True)
    if cfg.get('executor_python'):os.environ['DM_EXECUTOR_PYTHON']=cfg['executor_python']
    n=int(cfg.get('num_shards',1));items=[it for it in select_items(cfg,split_dir) if shard_of(it.id,n)==shard_index]
    status = {'shard':shard_index, 'stage':'startup', 'total':len(items), 'completed':0}
    status_warning = False
    def report(**values):
        nonlocal status_warning
        status.update(values, updated_epoch=time.time())
        try:
            atomic_write_json(out/f'status.shard{shard_index}.json', status)
        except (OSError, TypeError, ValueError) as error:
            if not status_warning:
                status_warning = True
                print(f'[shard {shard_index}] monitoring status unavailable: {error}', flush=True)
    report()
    print(f'[shard {shard_index}] starting mode={cfg["mode"]} items={len(items)}', flush=True)
    rh=run_hash(cfg)
    preds=DurableJsonl(out/f'preds.shard{shard_index}.jsonl',rh)
    traces=DurableJsonl(out/f'traces.shard{shard_index}.jsonl',rh)
    stages=DurableJsonl(out/f'states.shard{shard_index}.jsonl',rh)
    # Recover a crash between durable trace and prediction writes without re-executing that final answer.
    for qid,tr in traces.records.items():
        if tr.get('status')=='completed' and qid not in preds.done_ids() and tr.get('final'):
            preds.append({'id':qid,'status':'completed','record':tr['final'],'execution_id':tr.get('execution_id')})
    done=preds.done_ids();todo=sorted([it for it in items if it.id not in done],key=lambda it:(it.business_id,it.id))
    report(stage='verify_cache', completed=len(done), pending=len(todo))
    acfg=AgentConfig(mode=cfg['mode'],**cfg.get('agent',{}))
    shops=ShopContexts(split_dir,acfg.mode,Path(cfg['cache_dir'])/cfg['split'] if acfg.mode=='helper' else None,acfg.context_chars)
    if acfg.mode=='helper':
        from .profiler import cache_expectation,verify_shop_cache
        from .common import sha256_file
        for shop in sorted({it.business_id for it in todo}):
            source=split_dir/'data'/shop
            verify_shop_cache(source,shops.cache_dir/shop,cache_expectation(source,cfg['split'],sha256_file(split_dir/'rulebook.md')))
    rules=Rulebook((split_dir/'rulebook.md').read_text())
    monitor=MemoryMonitor();t=time.time();load_s=0.;completed=0;generation_and_execution_s=0.
    try:
        if todo and can_start(deadline,cfg.get('batch_estimate_s',0)):
            if llm is None:
                from .llm import make_llm
                report(stage='load_model')
                print(f'[shard {shard_index}] loading model={cfg["llm"].get("model")} backend={cfg["llm"].get("backend")}', flush=True)
                llm=make_llm(cfg['llm'])
            load_s=time.time()-t
            report(stage='model_ready', model_load_s=load_s, backend=type(llm).__name__)
            print(f'[shard {shard_index}] model ready backend={type(llm).__name__} load_s={load_s:.1f}', flush=True)
            from .monitoring import ObservedLLM
            llm=ObservedLLM(llm,report)
        def progress(event):
            nonlocal completed
            qid=event['id'];trace={**event['trace'],'execution_id':cfg.get('execution_id')}
            if event['stage']=='completed':
                traces.append({'id':qid,'status':'completed',**trace})
                preds.append({'id':qid,'status':'completed','record':event['record'],'execution_id':cfg.get('execution_id')})
                completed+=1
            stages.append({'id':qid,'status':'checkpoint','event':{**event,'trace':trace},'execution_id':cfg.get('execution_id')})
            report(stage=event['stage'], completed=len(preds.done_ids()), pending=len(items)-len(preds.done_ids()), last_id=qid)
            if event['stage']=='completed':
                print(f'[shard {shard_index}] completed={len(preds.done_ids())}/{len(items)} id={qid}', flush=True)
        bs=int(cfg.get('batch_size',16));estimate=float(cfg.get('batch_estimate_s',0));stopped=False
        for start in range(0,len(todo),bs):
            if not can_start(deadline,estimate):stopped=True;break
            batch=todo[start:start+bs]
            resume={i:r['event'] for i,r in stages.records.items()}
            began=time.time()
            report(stage='solve_batch', batch_size=len(batch), batch_start=start)
            print(f'[shard {shard_index}] batch started items={len(batch)} completed={len(preds.done_ids())}/{len(items)}', flush=True)
            solve_batch(batch,llm,rules,shops,acfg,on_progress=progress,resume_states=resume)
            elapsed=time.time()-began;generation_and_execution_s+=elapsed;estimate=max(estimate,elapsed*1.25)
        summary={'shard':shard_index,'run_hash':rh,'items_in_shard':len(items),'already_done':len(done),'completed_now':completed,
                 'remaining':len(set(it.id for it in items)-preds.done_ids()),'stopped_early_by_wallclock':stopped,
                 'model_load_s':load_s,'run_s':generation_and_execution_s,'s_per_item':generation_and_execution_s/completed if completed else None,
                 'peak_vram_mb':monitor.close(),'execution_id':cfg.get('execution_id'),'env':env_info()}
        report(stage='stopped' if stopped else 'finished', completed=len(preds.done_ids()), pending=summary['remaining'])
        atomic_write_json(out/f'summary.shard{shard_index}.json',summary);return summary
    finally:preds.close();traces.close();stages.close();monitor.close()


def launch(cfg,cfg_path,gpus=None,lib_root=None,python=None):
    out=Path(cfg['out_dir']);out.mkdir(parents=True,exist_ok=True)
    n=int(cfg.get('num_shards',1));gpus=list(range(n)) if gpus is None else gpus;processes=[]
    for i in range(n):
        env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpus[i%len(gpus)]) if gpus else '',PYTHONUNBUFFERED='1',PYTHONDONTWRITEBYTECODE='1')
        if lib_root:env['PYTHONPATH']=str(lib_root)+os.pathsep+env.get('PYTHONPATH','')
        if cfg.get('executor_python'):env['DM_EXECUTOR_PYTHON']=cfg['executor_python']
        log=open(out/f'worker{i}.log','a');processes.append((subprocess.Popen([python or sys.executable,'-m','dmlib.run','--config',str(cfg_path),'--shard',str(i)],env=env,stdout=log,stderr=subprocess.STDOUT,start_new_session=True),log))
    from .common import wait_workers
    return wait_workers(processes,cfg)


def main(argv=None):
    ap=argparse.ArgumentParser();ap.add_argument('--config',required=True);ap.add_argument('--shard',type=int,default=0)
    args=ap.parse_args(argv);cfg=json.loads(Path(args.config).read_text());print(json.dumps(work(cfg,args.shard),ensure_ascii=False))

if __name__=='__main__':main()


In [ ]:
%%writefile /kaggle/working/lib/dmlib/merge.py
"""Canonical records are the only predictions used by export and scoring."""
from __future__ import annotations
import json
from collections import Counter,defaultdict
from pathlib import Path
from .answer import penalized_accuracy,validate_record,format_answer
from .common import atomic_write_json,atomic_write_text,dumps,load_jsonl,sha256_file
from .provenance import collect_shards


def canonical_records(recs:dict,ids:list[str],finalize:bool):
    if len(ids)!=len(set(ids)):raise ValueError('duplicate manifest IDs')
    if set(recs)-set(ids):raise ValueError('prediction IDs outside manifest')
    canonical={};invalid={};pending=[]
    for qid in ids:
        record=recs.get(qid)
        if record is None:
            pending.append(qid)
            if finalize:canonical[qid]={'id':qid,'type':'abstain'}
            continue
        error=validate_record(record)
        if record.get('id')!=qid:error='record ID mismatch'
        if error:
            invalid[qid]=error;canonical[qid]={'id':qid,'type':'abstain'}
        else:canonical[qid]=format_answer(qid,record['type'],record.get('answer'),reason=record.get('reason'))
    return canonical,{'pending':pending,'invalid':invalid}


def collect(out_dir,expected_hash=None,ids=None):
    return {i:r['record'] for i,r in collect_shards(out_dir,'preds.shard*.jsonl',expected_hash,ids).items() if r.get('status')=='completed'}


def merge(out_dir,split_dir,finalize=False,submission_name='submission.jsonl',ids=None,cfg=None):
    out_dir,split_dir=Path(out_dir),Path(split_dir)
    questions=load_jsonl(split_dir/'questions.jsonl');order=ids if ids is not None else [q['id'] for q in questions]
    if cfg is None and (out_dir/'config.json').exists():cfg=json.loads((out_dir/'config.json').read_text())
    expected_hash=None
    if cfg is not None:
        from .run import run_hash
        expected_hash=run_hash(cfg)
    recs=collect(out_dir,expected_hash,order);canonical,meta=canonical_records(recs,order,finalize)
    text=''.join(dumps(canonical[i])+'\n' for i in order if i in canonical)
    atomic_write_text(out_dir/'canonical_predictions.jsonl',text)
    traces=collect_shards(out_dir,'traces.shard*.jsonl',expected_hash,order)
    explicit=sum(r['type']=='abstain' and not traces.get(i,{}).get('abstain_reason') for i,r in canonical.items() if i in recs and i not in meta['invalid'])
    after_error=sum(r['type']=='abstain' and bool(traces.get(i,{}).get('abstain_reason')) for i,r in canonical.items() if i in recs and i not in meta['invalid'])
    report={'n_questions':len(order),'n_written':len(canonical),'pending':len(meta['pending']),
            'pending_ids_head':meta['pending'][:20],'baseline_complete':not meta['pending'],'finalized':finalize,
            'invalid_replaced_by_abstain':meta['invalid'],'invalid_replaced':len(meta['invalid']),
            'abstain_explicit':explicit,'abstain_after_error':after_error,
            'type_counts':dict(Counter(r['type'] for r in canonical.values())),
            'prediction_sha256':sha256_file(out_dir/'canonical_predictions.jsonl'),'submission':None}
    if not meta['pending'] or finalize:
        atomic_write_text(out_dir/submission_name,text);report['submission']=str(out_dir/submission_name)
    elif (out_dir/submission_name).exists():(out_dir/submission_name).unlink()
    if (split_dir/'answers.jsonl').exists():
        gold={r['id']:r for r in load_jsonl(split_dir/'answers.jsonl') if r['id'] in set(order)}
        if set(gold)!=set(order):raise ValueError('missing/unknown gold IDs')
        score=penalized_accuracy(canonical,gold);per_id=score.pop('per_id');report['score']=score
        shops={q['id']:q['business_id'] for q in questions};by_shop,by_type=defaultdict(list),defaultdict(list)
        per=[]
        for i,s in per_id.items():
            by_shop[shops[i]].append(s);by_type[gold[i]['type']].append(s)
            per.append({'id':i,'score':s,'shop':shops[i],'gold_type':gold[i]['type'],'pending':i in meta['pending']})
        def summary(v):return {'n':len(v),'pa':sum(v)/len(v) if v else None,'correct':v.count(1),'wrong':v.count(-1),'abstain':v.count(0)}
        report['by_shop']={k:summary(v) for k,v in sorted(by_shop.items())}
        report['by_gold_type']={k:summary(v) for k,v in sorted(by_type.items())}
        atomic_write_text(out_dir/'per_id_scores.jsonl',''.join(dumps(r)+'\n' for r in per))
    atomic_write_json(out_dir/'merge_report.json',report)
    return report


In [ ]:
%%writefile /kaggle/working/lib/dmlib/regression.py
"""CPU contracts for raw/helper: canonical scoring, repair resume and OS file policy."""
import json
import tempfile
import unittest
from pathlib import Path
from . import answer, merge, split

class DataMateRegression(unittest.TestCase):
    def test_canonical_scores_exported_abstain_for_invalid_boolean(self):
        records,meta=merge.canonical_records({'a':{'id':'a','type':'boolean','answer':1}},['a','b'],False)
        self.assertEqual(records['a'],{'id':'a','type':'abstain'})
        self.assertEqual(meta['pending'],['b'])
        self.assertEqual(answer.score(records['a'],{'id':'a','type':'boolean','answer':True}),0)

    def test_pa_literal_fixture(self):
        gold={str(i):{'id':str(i),'type':'number','answer':i} for i in range(4)}
        pred={'0':gold['0'],'1':gold['1'],'2':{'id':'2','type':'number','answer':9},'3':{'id':'3','type':'abstain'}}
        result=answer.penalized_accuracy(pred,gold)
        self.assertEqual(result['pa'],.25);self.assertEqual(result['coverage'],.75)
        self.assertAlmostEqual(result['accuracy_answered'],2/3)
        self.assertIsNone(answer.penalized_accuracy({},gold)['accuracy_answered'])

    def test_types_dates_numbers_and_table_set(self):
        self.assertIsNotNone(answer.validate_record({'id':'a','type':'date','answer':'2026-02-30'}))
        self.assertIsNotNone(answer.validate_record({'id':'a','type':'boolean','answer':1}))
        self.assertIsNotNone(answer.validate_record({'id':'a','type':'number','answer':float('nan')}))
        self.assertEqual(answer.score({'type':'number','answer':1.234},{'type':'number','answer':1.23}),1)
        g={'type':'table','answer':{'columns':['name','n'],'rows':[['A',1]]}}
        p={'type':'table','answer':{'columns':['n','name'],'rows':[[1,'A'],[1,'A']]}}
        self.assertEqual(answer.score(p,g),1)
        self.assertEqual(answer.score({'type':'list','answer':[1,2]},{'type':'list','answer':[2,1]}),-1)

    def test_shop_manifest_is_disjoint_and_contains_no_labels(self):
        qs=[{'id':str(i),'business_id':f's{i:02}','question':'q','as_of_date':'2026-01-01','answer':'SECRET'} for i in range(30)]
        m=split.prepare_dev_run({'subset':'dev_tune','manifest_seed':2026},qs)
        self.assertEqual(len(m['shops']['dev_tune']),6)
        self.assertEqual(len(m['shops']['final_holdout']),6)
        self.assertEqual(len(m['shops']['train_fit']),18)
        self.assertNotIn('SECRET',json.dumps(m))
        self.assertFalse(set(m['ids']['dev_tune']) & set(m['ids']['final_holdout']))

    def test_executor_enforces_actual_file_access(self):
        import pandas as pd
        from .executor import run_code
        with tempfile.TemporaryDirectory() as td:
            root=Path(td);shop=root/'shop';shop.mkdir()
            (shop/'data.csv').write_text('n\n2\n3\n');(root/'secret').write_text('LABEL')
            (shop/'leak').symlink_to(root/'secret')
            (root/'calendar.csv').write_text('date\n2026-01-01\n')
            context={'DATA_DIR':str(shop),'CALENDAR_PATH':str(root/'calendar.csv'),'AS_OF':'2026-01-01'}
            good=run_code("import pandas as pd\nsubmit('number',int(pd.read_csv(DATA_DIR+'/data.csv')['n'].sum()))",'q',context)
            self.assertTrue(good.ok,good.error);self.assertEqual(good.record['answer'],5)
            self.assertGreaterEqual(good.extra['policy']['abi'],3)
            for code in ["open(DATA_DIR+'/leak').read()", "open(DATA_DIR+'/data.csv','w').write('bad')", "__import__('dmlib.helpers')"]:
                bad=run_code(code+"\nsubmit('number',0)",'q',context)
                self.assertFalse(bad.ok,code)
                self.assertIn('PermissionError',bad.error)
            scratch=run_code("open('scratch.txt','w').write('ok')\nsubmit('boolean',True)",'q',context)
            self.assertTrue(scratch.ok,scratch.error)

    def test_repair_backend_error_stays_pending_and_resumes_only_repair(self):
        from .agent import AgentConfig,Item,ShopContexts,solve_batch
        from .rulebook import Rulebook
        from .llm import GenOut
        with tempfile.TemporaryDirectory() as td:
            root=Path(td);(root/'data'/'shop').mkdir(parents=True)
            (root/'calendar.csv').write_text('date\n2026-01-01\n')
            items=[Item('good','shop','q','2026-01-01'),Item('bad','shop','q','2026-01-01')]
            class Backend:
                def __init__(self):self.calls=0
                def generate(self,convs,max_tokens):
                    self.calls+=1
                    if self.calls==1:return [GenOut("submit('number',1)",1,1,'stop'),GenOut("submit('number',missing)",1,1,'stop')]
                    raise RuntimeError('backend offline')
            events=[];shops=ShopContexts(root,'raw',None,9000);cfg=AgentConfig(mode='raw')
            out=solve_batch(items,Backend(),Rulebook('rules'),shops,cfg,on_progress=events.append)
            self.assertEqual(out[0].status,'completed')
            self.assertEqual(out[1].status,'retryable')
            self.assertEqual(out[1].trace['stage'],'repair_pending')
            self.assertTrue(any(e['id']=='good' and e['stage']=='completed' for e in events))
            saved={e['id']:e for e in events}
            class Repair:
                def generate(self,convs,max_tokens):
                    if len(convs)!=1 or convs[0][-2]['content']!="submit('number',missing)":raise AssertionError('first generation repeated')
                    return [GenOut("submit('number',2)",1,1,'stop')]
            resumed=solve_batch(items[1:],Repair(),Rulebook('rules'),shops,cfg,resume_states=saved)
            self.assertEqual(resumed[0].status,'completed');self.assertEqual(resumed[0].record['answer'],2)

    def test_cache_fingerprint_rejects_changed_input_and_artifact(self):
        from .profiler import profile_shop,cache_expectation,verify_shop_cache
        with tempfile.TemporaryDirectory() as td:
            root=Path(td);shop=root/'shop';shop.mkdir();(shop/'sales.csv').write_text('Mã đơn,Tiền\na,100\nb,200\n')
            profile_shop(shop,root/'cache',split='train',rulebook_sha='rules')
            cache=root/'cache'/'shop';expected=cache_expectation(shop,'train','rules')
            self.assertEqual(verify_shop_cache(shop,cache,expected)['schema'],'dm-profile-v1')
            (cache/'card.md').write_text('corrupt')
            with self.assertRaises(ValueError):verify_shop_cache(shop,cache,expected)
            (shop/'sales.csv').write_text('changed')
            with self.assertRaises(ValueError):verify_shop_cache(shop,cache,expected)

    def test_merge_full_denominator_and_conflicting_shards(self):
        from .common import DurableJsonl
        with tempfile.TemporaryDirectory() as td:
            root=Path(td);sd=root/'train';sd.mkdir();out=root/'out';out.mkdir()
            qs=[{'id':str(i),'business_id':'s'} for i in range(4)]
            (sd/'questions.jsonl').write_text(''.join(json.dumps(q)+'\n' for q in qs))
            (sd/'answers.jsonl').write_text(''.join(json.dumps({'id':str(i),'type':'number','answer':i})+'\n' for i in range(4)))
            w=DurableJsonl(out/'preds.shard0.jsonl','same')
            for i in range(2):w.append({'id':str(i),'status':'completed','record':{'id':str(i),'type':'number','answer':i}})
            w.close()
            rep=merge.merge(out,sd,ids=[str(i) for i in range(4)])
            self.assertFalse(rep['baseline_complete']);self.assertEqual(rep['pending'],2)
            self.assertEqual(rep['score']['pa'],.5);self.assertIsNone(rep['submission'])
            w=DurableJsonl(out/'preds.shard1.jsonl','same')
            w.append({'id':'0','status':'completed','record':{'id':'0','type':'number','answer':99}});w.close()
            with self.assertRaises(ValueError):merge.merge(out,sd)

    def test_raw_helper_contract_rejects_unclassified_config(self):
        from .compare import comparison_contract,validate_contracts
        cfg={'mode':'helper','llm':{'model':'m','revision':'r'},'agent':{'max_new_tokens':10},'num_shards':2}
        manifest={'manifest_sha256':'man','input_sha256':'input'}
        helper=comparison_contract(cfg,manifest)
        raw=comparison_contract(dict(cfg,mode='raw'),manifest)
        validate_contracts(raw,helper)
        changed=comparison_contract(dict(cfg,mode='raw',llm={'model':'other','revision':'r'}),manifest)
        with self.assertRaises(ValueError):validate_contracts(changed,helper)
        with self.assertRaises(ValueError):comparison_contract(dict(cfg,unknown_switch=True),manifest)

    def test_worker_resumes_durable_completed_ids_and_partial_last_line(self):
        from .run import work
        from .llm import GenOut
        class LLM:
            def __init__(self):self.seen=0
            def generate(self,convs,max_tokens):self.seen+=len(convs);return [GenOut("submit('number',1)",1,1,'stop') for c in convs]
        with tempfile.TemporaryDirectory() as td:
            root=Path(td);sd=root/'train';(sd/'data'/'shop').mkdir(parents=True)
            (sd/'calendar.csv').write_text('date\n2026-01-01\n');(sd/'rulebook.md').write_text('rules')
            (sd/'questions.jsonl').write_text(''.join(json.dumps({'id':str(i),'business_id':'shop','question':'q','as_of_date':'2026-01-01'})+'\n' for i in range(3)))
            cfg={'split':'train','split_dir':str(sd),'subset':'all','out_dir':str(root/'out'),'cache_dir':str(root/'cache'),
                 'experiment':'test','mode':'raw','num_shards':1,'batch_size':2,'llm':{'backend':'mock'},'agent':{'repair':True}}
            llm=LLM();self.assertEqual(work(cfg,0,llm)['completed_now'],3);self.assertEqual(llm.seen,3)
            path=root/'out'/'preds.shard0.jsonl'
            with open(path,'a') as f:f.write('{"id":')
            self.assertEqual(work(dict(cfg,execution_id='continuation'),0,llm)['completed_now'],0)
            self.assertEqual(llm.seen,3)
            with self.assertRaises(RuntimeError):work(dict(cfg,num_shards=2),0,llm)

    def test_preflight_chooses_only_common_verified_config(self):
        from .preflight import choose_serving
        records=[{'trials':[{'capacity':1,'ok':True,'workload_s':10},{'capacity':2,'ok':True,'workload_s':9.8}]},
                 {'trials':[{'capacity':1,'ok':True,'workload_s':10},{'capacity':2,'ok':False,'workload_s':None}]}]
        self.assertEqual(choose_serving(records),1)
        records[1]['trials'][0]['ok']=False
        with self.assertRaises(RuntimeError):choose_serving(records)

    def test_dev_drivers_complete_both_modes_with_same_contract(self):
        from unittest.mock import patch
        from .experiments import preset,run_dev
        from .run import work
        from .llm import GenOut
        class LLM:
            def generate(self,convs,max_tokens):return [GenOut("submit('number',1)",1,1,'stop') for c in convs]
        with tempfile.TemporaryDirectory() as td:
            root=Path(td);sd=root/'train';sd.mkdir()
            (sd/'calendar.csv').write_text('date,lunar_day,lunar_month,lunar_year,lunar_leap\n2026-01-01,1,1,2026,0\n');(sd/'rulebook.md').write_text('Quy tắc chung')
            qs=[];golds=[]
            for i in range(30):
                shop=sd/'data'/f's{i:02}';shop.mkdir(parents=True);(shop/'sales.csv').write_text('Mã đơn,Tiền\na,100\nb,200\n')
                qs.append({'id':str(i),'business_id':shop.name,'question':'Có bao nhiêu?','as_of_date':'2026-01-01'})
                golds.append({'id':str(i),'type':'number','answer':1})
            for name,rows in [('questions',qs),('answers',golds)]:
                (sd/(name+'.jsonl')).write_text(''.join(json.dumps(r)+'\n' for r in rows))
            base={'llm':{},'agent':{},'cache_dir':str(root/'cache'),'batch_size':16,'cpu_threads':1,'seed':0}
            def launch(cfg,path,gpus,lib_root,python):
                return [0 if work(cfg,i,LLM()) is not None else 1 for i in range(2)]
            def serving(cfg,*args):return dict(cfg,admission_wave_size=1,preflight_sha256='fixed')
            with patch('dmlib.preflight.prepare_serving',serving),patch('dmlib.run.launch',launch):
                helper=preset(base,'baseline_dev');helper['out_dir']=str(root/'helper')
                a=run_dev(helper,sd,Path(__file__).parent.parent,[0,1],__import__('sys').executable)
                self.assertTrue(a['baseline_complete']);self.assertEqual(a['score']['pa'],1)
                raw=preset(base,'raw_dev');raw.update(out_dir=str(root/'raw'),preflight_dirs=[str(root/'helper')],control_dir=str(root/'helper'))
                # The raw flow must not invoke a profiler, including through preflight reuse.
                with patch('dmlib.experiments.prepare_caches',side_effect=AssertionError('raw built cache')):
                    b=run_dev(raw,sd,Path(__file__).parent.parent,[0,1],__import__('sys').executable)
                self.assertTrue(b['baseline_complete'])
                result=json.loads((root/'architecture_comparison.json').read_text())
                self.assertTrue(result['comparison_complete']);self.assertEqual(result['delta_pa'],0)

    def test_expired_session_does_not_load_model(self):
        from unittest.mock import patch
        from .run import work
        import time
        with tempfile.TemporaryDirectory() as td:
            root=Path(td);sd=root/'train';(sd/'data'/'shop').mkdir(parents=True)
            (sd/'calendar.csv').write_text('date\n2026-01-01\n');(sd/'rulebook.md').write_text('rules')
            (sd/'questions.jsonl').write_text(json.dumps({'id':'x','business_id':'shop','question':'q','as_of_date':'2026-01-01'})+'\n')
            cfg={'mode':'raw','split':'train','split_dir':str(sd),'subset':'all','out_dir':str(root/'out'),'cache_dir':str(root/'cache'),
                 'num_shards':1,'experiment':'deadline-test','llm':{},'session_deadline_epoch':time.time()-1,'export_reserve_s':900}
            with patch('dmlib.llm.make_llm',side_effect=AssertionError('model loaded after deadline')):
                report=work(cfg,0)
            self.assertEqual(report['completed_now'],0)

    def test_executor_reads_excel_sqlite_and_helper_cache(self):
        import pandas as pd
        import sqlite3
        from .executor import run_code
        from .profiler import profile_shop
        from .agent import PRELOAD_HELPER
        with tempfile.TemporaryDirectory() as td:
            root=Path(td);shop=root/'shop';shop.mkdir()
            pd.DataFrame({'n':[2,3]}).to_excel(shop/'sales.xlsx',index=False)
            con=sqlite3.connect(shop/'sales.sqlite');con.execute('create table sales(n integer)');con.executemany('insert into sales values(?)',[(2,),(3,)]);con.commit();con.close()
            (root/'calendar.csv').write_text('date\n2026-01-01\n')
            ctx={'DATA_DIR':str(shop),'CALENDAR_PATH':str(root/'calendar.csv'),'AS_OF':'2026-01-01'}
            code="import pandas as pd,sqlite3\na=int(pd.read_excel(DATA_DIR+'/sales.xlsx',engine='calamine')['n'].sum())\nc=sqlite3.connect('file:'+DATA_DIR+'/sales.sqlite?mode=ro',uri=True)\nb=c.execute('select sum(n) from sales').fetchone()[0]\nsubmit('number',a+b)"
            r=run_code(code,'q',ctx);self.assertTrue(r.ok,r.error);self.assertEqual(r.record['answer'],10)
            profile_shop(shop,root/'cache','train','rules')
            r=run_code("submit('number',len(T))",'q',{**ctx,'SHOP_CACHE':str(root/'cache'/'shop')},preload=PRELOAD_HELPER)
            self.assertTrue(r.ok,r.error);self.assertEqual(r.record['answer'],2)

    def test_llm_preserves_legacy_scheduler_defaults_and_forwards_strict_capacity(self):
        import types
        import sys
        from unittest.mock import patch
        from .llm import VLLM
        class Engine:
            def __init__(self,**kwargs):self.kwargs=kwargs
            def get_tokenizer(self):return object()
        module=types.SimpleNamespace(LLM=Engine,SamplingParams=object)
        with patch.dict(sys.modules,{'vllm':module}):
            legacy=VLLM(model='fixture')
            self.assertNotIn('max_num_seqs',legacy.llm.kwargs)
            strict=VLLM(model='fixture',max_num_seqs=2,max_num_batched_tokens=4096,enable_chunked_prefill=True)
            self.assertEqual(strict.llm.kwargs['max_num_seqs'],2)

    def test_smoke_rejects_completed_abstain_from_execution_failures(self):
        from unittest.mock import patch
        from .experiments import smoke_workload
        from .agent import Outcome
        with tempfile.TemporaryDirectory() as td:
            root=Path(td);(root/'questions.jsonl').write_text(json.dumps({'id':'x','business_id':'s','question':'q','as_of_date':'2026-01-01'})+'\n')
            (root/'rulebook.md').write_text('rules')
            cfg={'split_dir':str(root),'split':'train','mode':'helper','cache_dir':str(root/'cache'),'agent':{'context_chars':9000}}
            failure=Outcome({'id':'x','type':'abstain'},'completed',{'attempts':[{'ok':False,'policy':{'abi':8},'error':'broken preload'}]})
            with patch('dmlib.experiments.prepare_dev_run',return_value={'smoke_ids':['x']}),patch('dmlib.agent.solve_batch',return_value=[failure]):
                with self.assertRaises(RuntimeError):smoke_workload(object(),cfg,1)

    def test_comparison_rejects_changed_shared_generation_code(self):
        from unittest.mock import patch
        from . import compare
        cfg={'mode':'helper','llm':{'model':'m','revision':'r'},'agent':{},'num_shards':2}
        manifest={'manifest_sha256':'man','input_sha256':'input'}
        helper=compare.comparison_contract(cfg,manifest)
        original=compare.sha256_file
        def changed(path):return 'different-code' if Path(path).name=='llm.py' else original(path)
        with patch('dmlib.compare.sha256_file',changed):raw=compare.comparison_contract(dict(cfg,mode='raw'),manifest)
        with self.assertRaises(ValueError):compare.validate_contracts(raw,helper)

    def test_launcher_streams_worker_output_before_worker_exits(self):
        import contextlib,io,subprocess,sys,time
        from .common import wait_workers
        with tempfile.TemporaryDirectory() as td:
            log=open(Path(td)/'worker0.log','a')
            process=subprocess.Popen([sys.executable,'-u','-c',"import time;print('live-marker',flush=True);time.sleep(.3)"],stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
            class Capture(io.StringIO):
                saw_live=False
                def write(self,text):
                    if 'live-marker' in text and process.poll() is None:self.saw_live=True
                    return super().write(text)
            capture=Capture()
            with contextlib.redirect_stdout(capture):
                codes=wait_workers([(process,log)],{'out_dir':td,'heartbeat_s':.05,'worker_poll_s':.01})
            self.assertEqual(codes,[0])
            self.assertTrue(capture.saw_live,'worker output remained hidden until exit')
            self.assertIn('heartbeat',capture.getvalue())

    def test_generation_monitor_preserves_outputs_and_logs_failures(self):
        import contextlib,io
        from .monitoring import ObservedLLM
        from .llm import GenOut
        state=[]
        class LLM:
            def generate(self,convs,max_tokens):
                if max_tokens==0:raise RuntimeError('backend fixture error')
                return [GenOut('reply',3,2,'stop') for c in convs]
        model=ObservedLLM(LLM(),lambda **kw:state.append(kw))
        with contextlib.redirect_stdout(io.StringIO()) as capture:
            result=model.generate([[{'role':'user','content':'fixture'}]],10)
            with self.assertRaises(RuntimeError):model.generate([[]],0)
        self.assertEqual(result[0].text,'reply')
        self.assertEqual(result[0].output_tokens,2)
        self.assertEqual(state[0]['stage'],'generation')
        self.assertEqual(state[-1]['stage'],'generation_error')
        self.assertIn('generation finished',capture.getvalue())
        self.assertIn('backend fixture error',capture.getvalue())

    def test_telemetry_failure_does_not_discard_generation(self):
        import contextlib,io
        from .monitoring import ObservedLLM
        from .llm import GenOut
        class LLM:
            def generate(self,convs,max_tokens):return [GenOut('kept',3,2,'stop')]
        def broken_report(**kwargs):raise OSError('status disk unavailable')
        with contextlib.redirect_stdout(io.StringIO()):
            result=ObservedLLM(LLM(),broken_report).generate([[]],5)
        self.assertEqual(result[0].text,'kept')

    def test_worker_finishing_at_deadline_keeps_success_exit_code(self):
        import contextlib,io,time
        from unittest.mock import patch
        from .common import wait_workers
        class Process:
            pid=123456789
            returncode=None
            def poll(self):return self.returncode
            def wait(self,**kwargs):self.returncode=0;return 0
        with tempfile.TemporaryDirectory() as td:
            log=open(Path(td)/'worker.log','a');process=Process()
            with contextlib.redirect_stdout(io.StringIO()),patch('os.killpg',side_effect=ProcessLookupError):
                codes=wait_workers([(process,log)],{'session_deadline_epoch':time.time()-1,'export_reserve_s':0})
            self.assertEqual(codes,[0])

if __name__=='__main__':unittest.main()


In [ ]:
%%writefile /kaggle/working/lib/dmlib/provenance.py
"""Content fingerprints and strict shard collection; execution metadata is separate."""
from __future__ import annotations
import hashlib
import inspect
import json
import shutil
from pathlib import Path
from .common import atomic_write_json, read_jsonl_tolerant, sha256_file

RUNTIME_KEYS = {'execution_id','parent_execution_id','account_alias','kaggle_version','timestamp','out_dir','input_roots','split_dir','cache_dir','asr_cache_dirs','resume_from_dirs','resume','wallclock_s','session_wallclock_s','session_started_epoch','session_deadline_epoch','export_reserve_s','batch_estimate_s','preflight_dirs','control_dir','candidate_dir','lib_root','llm_python','executor_python','num_gpus','shard_index','notes','parent_run','source_execution','source_version','asr_expectation','code_sha256','verify_reuse','asr_reused_from','heartbeat_s','worker_poll_s','worker_phase'}

def digest(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,ensure_ascii=False,allow_nan=False,default=str,separators=(',',':')).encode()).hexdigest()

def source_hash(obj):
    return hashlib.sha256(inspect.getsource(obj).encode()).hexdigest()

def package_hash(package_dir):
    return digest({p.name:sha256_file(p) for p in sorted(Path(package_dir).glob('*.py')) if p.name not in {'regression.py'}})

def tree_hash(directory):
    root=Path(directory).resolve()
    records={}
    for p in sorted(root.rglob('*')):
        if p.is_file():
            if not p.resolve().is_relative_to(root):raise ValueError(f'source symlink escapes root: {p}')
            records[str(p.relative_to(root))]=sha256_file(p)
    return digest(records)

def semantic_hash(cfg):
    semantic={k:v for k,v in cfg.items() if k not in RUNTIME_KEYS}
    if cfg.get('experiment_id') and cfg.get('experiment') != cfg['experiment_id']:
        raise ValueError('experiment and experiment_id must agree')
    # Interpreter and source identities are content-addressed in preflight/manifest, never by mount path.
    semantic['code_sha256']=cfg.get('code_sha256') or package_hash(Path(__file__).parent)
    return digest(semantic)

def freeze_json(path,value):
    p=Path(path)
    if p.exists() and json.loads(p.read_text()) != value:raise ValueError(f'frozen artifact differs: {p}')
    atomic_write_json(p,value)
    return value

def collect_shards(out_dir,pattern,expected_hash=None,allowed_ids=None):
    records={};expected=set(allowed_ids) if allowed_ids is not None else None
    for p in sorted(Path(out_dir).glob(pattern)):
        rows,_=read_jsonl_tolerant(p)
        header=rows[0].get('_header') if rows else None
        if not header or (expected_hash is not None and header.get('run_hash')!=expected_hash):
            raise ValueError(f'shard header mismatch: {p}')
        if expected_hash is None:expected_hash=header.get('run_hash')
        for rec in rows[1:]:
            qid=rec.get('id')
            if expected is not None and qid not in expected:raise ValueError(f'unknown ID {qid} in {p}')
            previous=records.get(qid)
            if previous and previous.get('status')=='completed':
                if rec.get('status')!='completed' or digest({k:v for k,v in previous.items() if k not in {'execution_id'}})!=digest({k:v for k,v in rec.items() if k not in {'execution_id'}}):
                    raise ValueError(f'conflicting completed ID: {qid}')
            records[qid]=rec
    return records

def restore_outputs(dirs,destination):
    """Copy explicitly attached checkpoint roots, never scan all inputs or overwrite another lineage."""
    dest=Path(destination);dest.mkdir(parents=True,exist_ok=True)
    for directory in dirs:
        src=Path(directory)
        if not src.is_dir():raise FileNotFoundError(src)
        for p in src.rglob('*'):
            if not p.is_file() or p.suffix not in {'.json','.jsonl'}:continue
            target=dest/p.relative_to(src)
            if target.exists():
                if target.read_bytes()!=p.read_bytes():
                    if p.name in {'environment.json','wheels_manifest.json'}:continue
                    raise ValueError(f'resume destination already differs: {target}')
            else:
                target.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(p,target)


In [ ]:
%%writefile /kaggle/working/lib/dmlib/fs_guard.py
"""Unprivileged Landlock allowlist, installed in a fresh single-threaded child.

Requires ABI >= 3 (including truncate restrictions). No broad input/home/proc grants.
API reference: https://docs.kernel.org/userspace-api/landlock.html
"""
from __future__ import annotations
import ctypes
import errno
import hashlib
import json
import os
import platform
import sys
import sysconfig
from pathlib import Path


def _libc():
    if platform.system() != 'Linux' or platform.machine() not in ('x86_64','aarch64'):
        raise RuntimeError('Landlock unavailable on this OS/architecture')
    return ctypes.CDLL(None,use_errno=True)


def _check(result,operation):
    if result < 0:
        err=ctypes.get_errno()
        raise RuntimeError(f'Landlock {operation}: {os.strerror(err)} (errno {err})')
    return result


def landlock_abi():
    return _check(_libc().syscall(444,0,0,1),'query ABI')


def runtime_read_paths(mode):
    paths={sysconfig.get_path(k) for k in ('stdlib','platstdlib','purelib','platlib')}
    paths.update(['/usr/lib','/usr/lib64','/lib','/lib64','/usr/share/zoneinfo',
                  '/etc/ld.so.cache','/etc/localtime','/dev/null','/dev/urandom','/sys/devices/system/cpu'])
    package=Path(__file__).resolve().parent
    modules=['__init__','answer','parsers']
    if mode=='helper':modules+=['common','loader','profiler','helpers','periods','provenance']
    paths.update(str(package/(name+'.py')) for name in modules)
    # Python's prefix may differ from stdlib (uv-managed base interpreter + a venv).
    paths.add(str(Path(sys.base_prefix)/'lib'))
    return sorted(str(Path(p).resolve()) for p in paths if p and Path(p).exists())


def restrict_executor_files(read_paths: list[str],scratch_dir: str) -> dict:
    libc=_libc();abi=landlock_abi()
    if abi<3:raise RuntimeError(f'Landlock ABI {abi} cannot enforce required truncate policy (need >=3)')
    # Rights through TRUNCATE; supported on every ABI accepted here. Include TCP restrictions on ABI>=4.
    handled=(1<<15)-1
    class Ruleset(ctypes.Structure):
        _fields_=[('fs',ctypes.c_uint64),('net',ctypes.c_uint64)]
    class Rule(ctypes.Structure):
        _pack_=1
        _fields_=[('access',ctypes.c_uint64),('fd',ctypes.c_int32)]
    attr=Ruleset(handled,3 if abi>=4 else 0)
    ruleset=_check(libc.syscall(444,ctypes.byref(attr),16 if abi>=4 else 8,0),'create ruleset')
    scratch=Path(scratch_dir).resolve()
    paths=sorted({str(Path(p).resolve()) for p in read_paths})
    try:
        for name in paths+[str(scratch)]:
            p=Path(name)
            if not p.exists():raise RuntimeError(f'allowlist path does not exist: {p}')
            if name==str(scratch):rights=handled & ~1  # all file operations, never executable
            else:rights=(1<<2) | ((1<<3) if p.is_dir() else 0)  # READ_FILE/READ_DIR
            if not p.is_dir():rights &= (1<<1)|(1<<2)|(1<<14)
            fd=os.open(p,os.O_PATH|os.O_CLOEXEC)
            try:
                rule=Rule(rights,fd)
                _check(libc.syscall(445,ruleset,1,ctypes.byref(rule),0),'add path')
            finally:os.close(fd)
        _check(libc.prctl(38,1,0,0,0),'no_new_privs')
        _check(libc.syscall(446,ruleset,0),'restrict self')
    finally:os.close(ruleset)
    return {'abi':abi,'schema':'dm-landlock-v1','policy_digest':hashlib.sha256(json.dumps({'paths':paths,'scratch':'<scratch>','fs':handled,'net':attr.net},sort_keys=True).encode()).hexdigest()}


def probe():
    """Check allow-read, deny-read, deny-write, scratch-write in a disposable process; never landlock the notebook."""
    import subprocess
    import tempfile
    with tempfile.TemporaryDirectory(prefix='dm_guard_probe_') as td:
        root=Path(td);(root/'allowed').write_text('ok');(root/'secret').write_text('label');(root/'scratch').mkdir()
        script='''import json
from pathlib import Path
from dmlib.fs_guard import restrict_executor_files
root=Path(ROOT)
policy=restrict_executor_files([str(root/'allowed')],str(root/'scratch'))
assert (root/'allowed').read_text()=='ok'
for path,mode in [(root/'secret','r'),(root/'allowed','w')]:
    try:open(path,mode)
    except PermissionError:pass
    else:raise AssertionError('file policy was not enforced')
(root/'scratch'/'ok').write_text('yes')
print(json.dumps(policy))
'''.replace('ROOT',repr(str(root)))
        env=dict(os.environ,PYTHONPATH=str(Path(__file__).resolve().parent.parent),PYTHONDONTWRITEBYTECODE='1')
        result=subprocess.run([sys.executable,'-c',script],capture_output=True,text=True,env=env,timeout=20,close_fds=True)
        if result.returncode:raise RuntimeError('Landlock probe failed: '+result.stderr[-2000:])
        return json.loads(result.stdout)


In [ ]:
%%writefile /kaggle/working/lib/dmlib/analysis.py
"""Post-inference error groups; semantic errors require manual trace review."""
from collections import Counter,defaultdict
from pathlib import Path
import json
from .common import atomic_write_json,atomic_write_text,dumps,load_jsonl
from .provenance import collect_shards


def summarize_baseline(out_dir,manifest_path):
    root=Path(out_dir);manifest=json.loads(Path(manifest_path).read_text());ids=manifest['selected_ids']
    traces=collect_shards(root,'traces.shard*.jsonl',allowed_ids=ids)
    scores={r['id']:r for r in load_jsonl(root/'per_id_scores.jsonl')} if (root/'per_id_scores.jsonl').exists() else {}
    per=[]
    for qid in ids:
        tr=traces.get(qid,{});score=scores.get(qid,{});attempts=tr.get('attempts',[])
        error=str(tr.get('abstain_reason') or (attempts[-1].get('error') if attempts else '') or '')
        if score.get('pending') or not tr:group='pending'
        elif tr.get('loader_error'):group='loader'
        elif score.get('score')==1:group='correct'
        elif not tr.get('abstain_reason') and tr.get('final',{}).get('type')=='abstain':group='abstain_explicit'
        elif 'timeout' in error.lower():group='timeout'
        elif 'too long' in error:group='context_too_long'
        elif 'SyntaxError' in error:group='syntax'
        elif any(a.get('finish')=='length' for a in attempts):group='truncated_output'
        elif 'format' in error.lower() or 'AnswerError' in error:group='format'
        elif error:group='runtime'
        else:group='semantic_review_needed'
        per.append({'id':qid,'shop':tr.get('shop'),'group':group,'error':error})
    result={'n':len(ids),'groups':dict(Counter(r['group'] for r in per)),
            'examples':{g:[r['id'] for r in per if r['group']==g][:3] for g in {r['group'] for r in per}}}
    atomic_write_text(root/'error_groups.jsonl',''.join(dumps(r)+'\n' for r in per))
    atomic_write_json(root/'baseline_report.json',result);return result


In [ ]:
%%writefile /kaggle/working/lib/dmlib/compare.py
"""Explicit common/treatment contract and offline raw/helper comparison."""
from __future__ import annotations
import json
from collections import Counter
from pathlib import Path
from .common import atomic_write_json,atomic_write_text,dumps,load_jsonl,sha256_file
from .provenance import digest,source_hash,RUNTIME_KEYS

_COMMON={'track','split','subset','manifest_seed','limit','shops','llm','agent','batch_size','cpu_threads','seed','num_shards','strict_dev','manifest_sha256','input_sha256','serving','admission_wave_size','fs_policy','repair_policy','model_manifest','input_fingerprints','preflight_sha256'}
_TREATMENT={'mode','preset','experiment','experiment_id','cache_fingerprints'}
_OPERATIONAL={'finalize','install_vllm','vllm_spec','calibration_s','calibration_candidates','asr_backend','asr_inline','smoke_ids','profiler_cache_dirs','source_preset','wheels_sha256','preflight_only'}

def comparison_contract(cfg,manifest,component_hashes=None):
    from . import agent,executor,answer,rulebook
    unknown=set(cfg)-_COMMON-_TREATMENT-_OPERATIONAL-RUNTIME_KEYS
    if unknown:raise ValueError(f'unclassified config fields: {sorted(unknown)}')
    common={k:v for k,v in cfg.items() if k in _COMMON}
    common_components={'SYSTEM_COMMON':digest(agent.SYSTEM_COMMON),'executor':sha256_file(Path(executor.__file__)),
                       'formatter_scorer':sha256_file(Path(answer.__file__)), 'rules':sha256_file(Path(rulebook.__file__)),
                       'solve_batch':source_hash(agent.solve_batch),'build_messages':source_hash(agent.build_messages),
                       'repair':source_hash(agent.repair_messages),'fs_guard':sha256_file(Path(__file__).with_name('fs_guard.py'))}
    for name in ('common','provenance','parsers','llm','run','merge','split'):
        common_components[name]=sha256_file(Path(__file__).with_name(name+'.py'))
    for name in ('shop_dir','exec_context','industry_text','text'):
        common_components['ShopContexts.'+name]=source_hash(getattr(agent.ShopContexts,name))
    treatment={'mode':cfg['mode'],'prompt':digest(agent.SYSTEM_RAW if cfg['mode']=='raw' else agent.SYSTEM_HELPER),
               'preload':digest(agent.PRELOAD_RAW if cfg['mode']=='raw' else agent.PRELOAD_HELPER),
               'context':source_hash(agent.raw_shop_context if cfg['mode']=='raw' else agent.helper_shop_context),
               'cache_fingerprints':cfg.get('cache_fingerprints') if cfg['mode']=='helper' else None}
    return {'schema':'dm-architecture-v1','mode':cfg['mode'],
            'common':digest({'cfg':common,'components':common_components,'input':manifest['input_sha256'],'manifest':manifest['manifest_sha256'],'extra':component_hashes or {}}),
            'treatment':treatment,'common_components':common_components}


def validate_contracts(raw,helper):
    if raw.get('schema')!='dm-architecture-v1' or helper.get('schema')!='dm-architecture-v1':raise ValueError('unknown comparison contract')
    if raw.get('mode')!='raw' or helper.get('mode')!='helper':raise ValueError('wrong architecture treatment')
    if raw.get('common')!=helper.get('common'):raise ValueError('raw/helper common fingerprint differs')


def compare_architectures(raw_dir,helper_dir,manifest_path):
    raw,helper=Path(raw_dir),Path(helper_dir);ids=json.loads(Path(manifest_path).read_text())['selected_ids']
    result={'comparison_complete':False,'n':len(ids)}
    if not all((d/'comparison_contract.json').is_file() and (d/'merge_report.json').is_file() for d in (raw,helper)):
        result['waiting_for']='raw/helper output';atomic_write_json(raw.parent/'architecture_comparison.json',result);return result
    contracts=[json.loads((d/'comparison_contract.json').read_text()) for d in (raw,helper)];validate_contracts(*contracts)
    records=[]
    for d in (raw,helper):
        report=json.loads((d/'merge_report.json').read_text())
        if not report.get('baseline_complete'):atomic_write_json(raw.parent/'architecture_comparison.json',result);return result
        if report['prediction_sha256']!=sha256_file(d/'canonical_predictions.jsonl'):raise ValueError('canonical predictions changed')
        rows=load_jsonl(d/'per_id_scores.jsonl');by_id={r['id']:r for r in rows}
        if len(rows)!=len(by_id) or set(by_id)-set(ids):raise ValueError('duplicate/unknown comparison IDs')
        if set(by_id)!=set(ids):atomic_write_json(raw.parent/'architecture_comparison.json',result);return result
        records.append(by_id)
    per=[{'id':i,'raw':records[0][i]['score'],'helper':records[1][i]['score'],'delta':records[0][i]['score']-records[1][i]['score'],
          'shop':records[0][i]['shop'],'gold_type':records[0][i]['gold_type']} for i in ids]
    def stats(rows):
        return {'n':len(rows),'delta_pa':sum(r['delta'] for r in rows)/len(rows) if rows else None,
                'transitions':dict(Counter(f"{r['helper']}->{r['raw']}" for r in rows))}
    result.update(comparison_complete=True,**stats(per),
                  by_shop={s:stats([r for r in per if r['shop']==s]) for s in sorted({r['shop'] for r in per})},
                  by_gold_type={t:stats([r for r in per if r['gold_type']==t]) for t in sorted({r['gold_type'] for r in per})})
    atomic_write_text(raw.parent/'architecture_per_id.jsonl',''.join(dumps(r)+'\n' for r in per))
    atomic_write_json(raw.parent/'architecture_comparison.json',result)
    return result


In [ ]:
%%writefile /kaggle/working/lib/dmlib/experiments.py
"""Independent raw/helper dev presets with shared input, runtime and scoring contracts."""
from __future__ import annotations
import copy
import json
import sys
import time
from pathlib import Path
from .common import atomic_write_json,load_jsonl,sha256_file
from .provenance import digest,tree_hash,package_hash,freeze_json,restore_outputs
from .models import MODELS
from .split import prepare_dev_run


def preset(base,name):
    cfg=copy.deepcopy(base)
    cfg.update(preset=name,strict_dev=True,split='train',subset='dev_tune',limit=None,manifest_seed=2026,
               session_wallclock_s=14400,wallclock_s=14400,export_reserve_s=900,num_shards=2,finalize=False,
               calibration_candidates=[1,2,4],preflight_dirs=[],profiler_cache_dirs=[],control_dir=None,
               fs_policy='dm-landlock-v1',repair_policy='durable-one-repair-v1')
    cfg['llm'].update(model='Qwen/Qwen3-4B-Instruct-2507',revision=MODELS['Qwen/Qwen3-4B-Instruct-2507']['sha'],
                      dtype='half',load_in_4bit=False,enable_thinking=False,max_model_len=16384,seed=0)
    cfg['agent'].update(max_new_tokens=1536,exec_timeout_s=60,exec_mem_mb=6144,exec_workers=1,repair=True,rules_k=8,context_chars=9000)
    if name=='baseline_dev':cfg.update(experiment='dm-D-001-baseline-dev',mode='helper',out_dir='/kaggle/working/dm_baseline')
    elif name=='raw_dev':cfg.update(experiment='dm-C-003-raw-dev',mode='raw',out_dir='/kaggle/working/dm_raw_baseline')
    else:raise ValueError(name)
    cfg.update(experiment_id=cfg['experiment'],execution_id=cfg['experiment'],parent_execution_id=None)
    return cfg


def prepare_caches(cfg,sd,shops):
    from .profiler import profile_shop,cache_expectation,verify_shop_cache
    fingerprints={};locations={};rb=sha256_file(sd/'rulebook.md')
    cache_root=Path(cfg['cache_dir'])/cfg['split']
    for shop in shops:
        print(f'[cache] checking {shop}', flush=True)
        started=time.time()
        source=sd/'data'/shop;expected=cache_expectation(source,cfg['split'],rb)
        target=cache_root/shop;found=None
        for root in cfg.get('profiler_cache_dirs',[]):
            candidate=Path(root)/cfg['split']/shop
            try:verify_shop_cache(source,candidate,expected);found=candidate;break
            except (ValueError,FileNotFoundError):pass
        if found is None:
            try:verify_shop_cache(source,target,expected)
            except (ValueError,FileNotFoundError):
                print(f'[cache] building {shop}', flush=True)
                profile_shop(source,cache_root,cfg['split'],rb,force=True)
            found=target
        # Worker cache roots must be one tree, not a mixture of mounts. Copy verified artifacts to working scratch.
        if found!=target:
            import shutil
            if target.exists():shutil.rmtree(target)
            target.parent.mkdir(parents=True,exist_ok=True);shutil.copytree(found,target)
        verified=verify_shop_cache(source,target,expected);fingerprints[shop]=digest(verified);locations[shop]=str(target)
        print(f'[cache] ready {shop} seconds={time.time()-started:.1f}', flush=True)
    return fingerprints


def smoke_workload(llm,cfg,capacity,short=False):
    from .agent import AgentConfig,ShopContexts,solve_batch
    from .run import select_items
    from .rulebook import Rulebook
    sd=Path(cfg['split_dir']);manifest=prepare_dev_run(cfg,load_jsonl(sd/'questions.jsonl'))
    ids=manifest['smoke_ids'][:2] if short else manifest['smoke_ids']
    items=select_items(dict(cfg,subset=ids),sd)
    if not items:raise RuntimeError('no train_fit smoke questions')
    rules=Rulebook((sd/'rulebook.md').read_text());result={}
    for mode in ([cfg['mode']] if cfg.get('verify_reuse') else ('raw','helper')):
        shops=ShopContexts(sd,mode,Path(cfg['cache_dir'])/cfg['split'] if mode=='helper' else None,cfg['agent']['context_chars'])
        print(f'[smoke] mode={mode} questions={len(items)} started', flush=True)
        acfg=AgentConfig(mode=mode,**cfg['agent']);t=time.time();out=solve_batch(items,llm,rules,shops,acfg)
        print(f'[smoke] mode={mode} finished seconds={time.time()-t:.1f}', flush=True)
        if any(o.status!='completed' for o in out):raise RuntimeError(f'{mode} smoke backend failed')
        if not any(a.get('ok') and a.get('policy') for o in out for a in o.trace.get('attempts',[])):
            raise RuntimeError(f'{mode} smoke produced no executed code; runtime not verified')
        result[mode]={'n':len(items),'wall_s':time.time()-t,'completed':sum(o.status=='completed' for o in out)}
    return result


def run_dev(cfg,split_dir,lib_root,gpus,llm_python,wheels_sha256=None):
    from .fs_guard import probe
    from .run import select_items,launch
    from .merge import merge
    from .preflight import prepare_serving
    from .compare import comparison_contract,compare_architectures
    from .analysis import summarize_baseline
    cfg=copy.deepcopy(cfg);sd=Path(split_dir);out=Path(cfg['out_dir']);out.mkdir(parents=True,exist_ok=True)
    cfg.update(split_dir=str(sd),executor_python=sys.executable,code_sha256=package_hash(Path(__file__).parent),wheels_sha256=wheels_sha256)
    cfg.setdefault('session_deadline_epoch',time.time()+cfg['session_wallclock_s'])
    restore_outputs(cfg.get('resume_from_dirs',[]),out)
    manifest=prepare_dev_run(cfg,load_jsonl(sd/'questions.jsonl'));freeze_json(out/'manifest.json',manifest)
    cfg.update(manifest_sha256=manifest['manifest_sha256'],input_sha256=manifest['input_sha256'])
    selected=select_items(dict(cfg,subset=manifest['selected_ids']),sd);shops=sorted({it.business_id for it in selected})
    smoke_items=select_items(dict(cfg,subset=manifest['smoke_ids']),sd);smoke_shops=sorted({it.business_id for it in smoke_items})
    cfg['input_fingerprints']={'rulebook':sha256_file(sd/'rulebook.md'),'calendar':sha256_file(sd/'calendar.csv'),
                               'shops':{s:tree_hash(sd/'data'/s) for s in sorted(set(shops+smoke_shops))}}
    cfg['model_manifest']=MODELS
    # Guard before profile/model load, including the interpreter that will execute generated code.
    atomic_write_json(out/'fs_guard_probe.json',probe())
    cfg['llm']['backend']='vllm' if str(llm_python)!=sys.executable else 'hf'
    # Shared smoke proves both architectures. A raw run with reusable preflight skips all profiler work.
    if cfg['mode']=='helper':
        cfg['cache_fingerprints']=prepare_caches(cfg,sd,sorted(set(shops+smoke_shops)))
    elif not cfg.get('preflight_dirs'):
        raise ValueError('raw_dev needs an attached verified preflight from baseline_dev; raw inference never builds helper caches')
    cfg=prepare_serving(cfg,str(llm_python),str(lib_root),gpus)
    if cfg.get('preflight_only'):
        return {'preflight_complete':True,'baseline_complete':False,'stage':'preflight_only'}
    cfg['subset']=manifest['selected_ids']
    contract=comparison_contract(cfg,manifest);freeze_json(out/'comparison_contract.json',contract)
    atomic_write_json(out/'config.json',cfg)
    codes=launch(cfg,str(out/'config.json'),gpus,str(lib_root),str(llm_python))
    report=merge(out,sd,finalize=False,ids=manifest['selected_ids'],cfg=cfg)
    summarize_baseline(out,out/'manifest.json')
    if cfg.get('control_dir'):
        helper=out if cfg['mode']=='helper' else Path(cfg['control_dir'])
        raw=out if cfg['mode']=='raw' else Path(cfg['control_dir'])
        compare_architectures(raw,helper,out/'manifest.json')
    if any(codes):raise RuntimeError(f'worker failed {codes}; partial checkpoint retained at {out}')
    return report


In [ ]:
%%writefile /kaggle/working/lib/dmlib/models.py
"""Public Hub metadata checked 2026-10-29; runtime counts are recorded separately."""
MODELS = {'Qwen/Qwen3-4B-Instruct-2507': {'sha': 'cdbee75f17c01a7cc42f958dc650907174af0554', 'createdAt': '2025-08-05T10:58:03.000Z', 'lastModified': '2025-09-17T06:56:53.000Z', 'safetensors': {'parameters': {'BF16': 4022468096}, 'total': 4022468096}, 'license': 'apache-2.0', 'files': [{'rfilename': 'model-00001-of-00003.safetensors', 'size': 3957900840, 'lfs': {'sha256': '75311d91bb08cf0b882913da464a1e722a31fb44db35208663487efb7a3d8ed6', 'size': 3957900840, 'pointerSize': 135}}, {'rfilename': 'model-00002-of-00003.safetensors', 'size': 3987450520, 'lfs': {'sha256': '0b48adbb1f60e901153d91907ba11ce63bd4b8b584482e730f48808d055dfba1', 'size': 3987450520, 'pointerSize': 135}}, {'rfilename': 'model-00003-of-00003.safetensors', 'size': 99630640, 'lfs': {'sha256': '7dd39ccca5e4de123c74c14af44c9bf2eb75df33b4614382af0134528e060d5d', 'size': 99630640, 'pointerSize': 133}}]}}


In [ ]:
%%writefile /kaggle/working/lib/dmlib/preflight.py
"""Freeze a serving config proven on both GPUs; never silently fall back during dev."""
from __future__ import annotations
import argparse
import gc
import importlib.metadata
import json
import os
import platform
import subprocess
import sys
import time
from pathlib import Path
from .common import atomic_write_json,sha256_file,MemoryMonitor
from .provenance import digest


def runtime_signature():
    packages={}
    for name in ('torch','transformers','vllm','pandas','pyarrow','python-calamine','numpy','soundfile','sherpa-onnx'):
        try:packages[name]=importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:packages[name]=None
    try:
        gpu=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],capture_output=True,text=True,timeout=10).stdout.strip().splitlines()
    except OSError:gpu=[]
    return {'python':platform.python_version(),'packages':packages,'gpu':gpu,'cpus':os.cpu_count()}


def choose_serving(records):
    available=None;scores={}
    for record in records:
        valid={r['capacity']:r['workload_s'] for r in record['trials'] if r.get('ok')}
        available=set(valid) if available is None else available & set(valid)
        for cap,seconds in valid.items():scores.setdefault(cap,[]).append(seconds)
    if not available:raise RuntimeError('no serving configuration passed on every GPU/mode')
    totals={c:sum(scores[c])/len(scores[c]) for c in available};best=min(totals.values())
    return min(c for c in available if totals[c]<=best*1.05)


def with_capacity(llm_cfg,capacity):
    out=dict(llm_cfg)
    if out['backend']=='vllm':out.update(max_num_seqs=capacity,max_num_batched_tokens=4096,enable_chunked_prefill=True)
    else:out['batch_size']=capacity
    return out


def long_prompt(llm,index,max_tokens):
    """Different prefixes prevent prefix-cache reuse masking KV pressure."""
    target=llm.max_model_len-max_tokens-24
    if target<=0:raise ValueError('no prompt token budget')
    prefix=f'Kiểm tra bộ nhớ độc lập {index}: '
    lo,hi=0,target*12
    while lo<hi:
        mid=(lo+hi+1)//2
        conv=[{'role':'user','content':prefix+(' từ'+str(index))*mid}]
        if llm.count_tokens(conv)<=target:lo=mid
        else:hi=mid-1
    conv=[{'role':'user','content':prefix+(' từ'+str(index))*lo}]
    return conv


def calibrate(cfg,shard):
    from .llm import make_llm
    from .experiments import smoke_workload
    track=cfg['track'];limit=min(time.time()+1200,cfg.get('calibration_deadline_epoch',float('inf')))
    capacities=cfg.get('calibration_candidates') or ([1,2,4] if cfg['llm']['backend']=='vllm' else [1,2,4,8])
    records=[];last_good=None
    for capacity in capacities:
        if time.time()>=limit:break
        llm=None;monitor=MemoryMonitor();trial={'capacity':capacity,'ok':False}
        try:
            print(f'[preflight {shard}] capacity={capacity} loading model', flush=True)
            start=time.time();llm=make_llm(with_capacity(cfg['llm'],capacity));trial['model_load_s']=time.time()-start
            print(f'[preflight {shard}] model ready load_s={trial["model_load_s"]:.1f}; testing long prompts', flush=True)
            start=time.time();tokens=cfg['agent']['max_new_tokens']
            prompts=[long_prompt(llm,i,tokens) for i in range(max(1,capacity))]
            outputs=llm.generate(prompts,tokens)
            if len(outputs)!=len(prompts) or any(o.finish in ('error','too_long') for o in outputs):raise RuntimeError('long-prompt smoke failed')
            trial['long_prompt_tokens']=[o.prompt_tokens for o in outputs]
            print(f'[preflight {shard}] long prompts passed; running train_fit smoke', flush=True)
            trial['workload']=smoke_workload(llm,cfg,capacity)
            trial.update(ok=True,workload_s=time.time()-start)
            last_good=capacity
        except Exception as e:
            trial.update(error=f'{type(e).__name__}: {e}',workload_s=None)
        finally:
            trial['peak_vram_mb']=monitor.close();records.append(trial)
            if llm is not None:del llm
            gc.collect()
            try:
                import torch
                torch.cuda.empty_cache()
            except ImportError:pass
        if not trial['ok']:break
    # A failed/OOM worker is discarded. Reverify the last good config in a fresh process before freeze.
    result={'trials':records,'runtime':runtime_signature(),'last_good':last_good,'shard':shard}
    atomic_write_json(Path(cfg['out_dir'])/f'calibration.shard{shard}.json',result)
    return result


def _spawn(cfg,python,lib_root,gpus,action='calibrate',timeout_s=None):
    out=Path(cfg['out_dir']);out.mkdir(parents=True,exist_ok=True);path=out/'preflight_config.json';atomic_write_json(path,cfg)
    processes=[];package=__package__
    for index,gpu in enumerate(gpus):
        env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu),PYTHONPATH=str(lib_root),PYTHONUNBUFFERED='1',PYTHONDONTWRITEBYTECODE='1')
        if cfg.get('executor_python'):env['DM_EXECUTOR_PYTHON']=cfg['executor_python']
        log=open(out/f'preflight{index}.{action}.log','w')
        process=subprocess.Popen([python,'-m',package+'.preflight','--config',str(path),'--shard',str(index),'--action',action],env=env,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
        processes.append((process,log))
    from .common import wait_workers
    wait_cfg = dict(cfg, worker_phase='preflight:'+action, session_deadline_epoch=time.time()+(timeout_s if timeout_s is not None else 1200), export_reserve_s=0)
    codes=wait_workers(processes,wait_cfg)
    if any(codes):raise RuntimeError(f'preflight workers failed {codes}; inspect {out}')


def prepare_serving(cfg,python,lib_root,gpus):
    if len(gpus)!=2:raise RuntimeError('dev presets require two GPUs; local regression never calls preflight')
    if cfg['llm']['backend'] not in ('hf','vllm') or not cfg['llm'].get('revision'):raise ValueError('resolve backend and model revision before preflight')
    out=Path(cfg['out_dir']);out.mkdir(parents=True,exist_ok=True)
    # Runtime is read using the actual worker interpreter, without allocating a GPU model.
    env=dict(os.environ,PYTHONPATH=str(lib_root),PYTHONDONTWRITEBYTECODE='1')
    script=f'from {__package__}.preflight import runtime_signature; import json; print(json.dumps(runtime_signature()))'
    current=json.loads(subprocess.check_output([python,'-c',script],env=env,text=True).splitlines()[-1])
    common={k:cfg.get(k) for k in ('track','manifest_sha256','input_sha256','code_sha256','input_fingerprints','model_manifest','agent','batch_size','cpu_threads','seed','wheels_sha256')}
    common['llm']={k:v for k,v in cfg['llm'].items() if k not in ('batch_size','max_num_seqs')}
    key=digest({'common':common,'runtime':current})
    for directory in [str(out),*cfg.get('preflight_dirs',[])]:
        p=Path(directory)/'preflight.json'
        if not p.is_file():continue
        old=json.loads(p.read_text())
        if old.get('compatibility_key')!=key:
            if Path(directory)==out:raise ValueError('existing preflight differs; use a new output lineage')
            continue
        fixed=dict(cfg,llm=old['llm'],admission_wave_size=old['capacity'],verify_reuse=True)
        _spawn(fixed,python,lib_root,gpus,action='verify',timeout_s=min(1200,max(1,cfg['session_deadline_epoch']-time.time()-cfg.get('export_reserve_s',900))))
        atomic_write_json(out/'preflight.json',{**old,'reused_from':str(p),'runtime':current})
        return {**fixed,'batch_estimate_s':old['batch_estimate_s'],'preflight_sha256':digest(old['llm'])}
    end=min(time.time()+1200,cfg['session_deadline_epoch']-cfg.get('export_reserve_s',900))
    capacities=cfg.get('calibration_candidates') or ([1,2,4] if cfg['llm']['backend']=='vllm' else [1,2,4,8])
    records=[{'trials':[]} for _ in gpus]
    for capacity in capacities:
        # Reserve a fresh-process confirmation before the global calibration deadline.
        if time.time()+90>=end:break
        calibration=dict(cfg,calibration_candidates=[capacity],calibration_deadline_epoch=end)
        _spawn(calibration,python,lib_root,gpus,timeout_s=max(1,end-time.time()-60))
        trials=[json.loads((out/f'calibration.shard{i}.json').read_text()) for i in range(len(gpus))]
        for result,trial in zip(records,trials):result['trials'].extend(trial['trials'])
        if any(not trial['trials'] or not all(t.get('ok') for t in trial['trials']) for trial in trials):break
    calibration=dict(cfg,calibration_deadline_epoch=end)
    capacity=choose_serving(records);fixed=dict(cfg,llm=with_capacity(cfg['llm'],capacity),admission_wave_size=capacity)
    # Reload a fresh worker for a short verification of the selected config, including after an OOM trial.
    remaining=min(calibration['calibration_deadline_epoch']-time.time(),cfg['session_deadline_epoch']-time.time()-cfg.get('export_reserve_s',900))
    if remaining<=0:raise RuntimeError('calibration budget exhausted before verification')
    _spawn(fixed,python,lib_root,gpus,action='verify',timeout_s=remaining)
    estimate=max(t['workload_s'] for r in records for t in r['trials'] if t.get('ok') and t['capacity']==capacity)*1.5
    report={'schema':'serving-preflight-v1','compatibility_key':key,'capacity':capacity,'llm':fixed['llm'],
            'batch_estimate_s':estimate,'runtime':current,'trials':records,'worker_python':python,
            'executor_python':cfg.get('executor_python'),'wheels_sha256':cfg.get('wheels_sha256')}
    atomic_write_json(out/'preflight.json',report)
    return {**fixed,'batch_estimate_s':estimate,'preflight_sha256':digest(fixed['llm'])}


def main():
    ap=argparse.ArgumentParser();ap.add_argument('--config',required=True);ap.add_argument('--shard',type=int,default=0);ap.add_argument('--action',default='calibrate')
    args=ap.parse_args();cfg=json.loads(Path(args.config).read_text())
    if args.action=='calibrate':calibrate(cfg,args.shard)
    else:
        from .llm import make_llm
        from .experiments import smoke_workload
        llm=make_llm(cfg['llm']);smoke_workload(llm,cfg,cfg['admission_wave_size'],short=True)

if __name__=='__main__':main()


In [ ]:
%%writefile /kaggle/working/lib/dmlib/monitoring.py
"""Inference telemetry only: preserve generation arguments, outputs and exceptions."""
from __future__ import annotations
import time


class ObservedLLM:
    def __init__(self, model, report):
        self.model = model
        self.report = report
        self.calls = 0
        self._report_error_shown = False

    def __getattr__(self, name):
        return getattr(self.model, name)

    @staticmethod
    def _log(message):
        try: print(message, flush=True)
        except (OSError, ValueError): pass

    def _report(self, **values):
        try:
            self.report(**values)
        except Exception as error:
            if not self._report_error_shown:
                self._report_error_shown = True
                self._log(f'[monitoring] status unavailable: {type(error).__name__}: {error}')

    def generate(self, convs, max_tokens):
        self.calls += 1
        started = time.monotonic()
        self._report(stage='generation', generation_batch=len(convs), generation_call=self.calls)
        self._log(f'[llm] generation started call={self.calls} requests={len(convs)} max_tokens={max_tokens}')
        try:
            result = self.model.generate(convs, max_tokens)
        except Exception as error:
            seconds = time.monotonic() - started
            self._report(stage='generation_error', generation_s=seconds, error=f'{type(error).__name__}: {error}')
            self._log(f'[llm] generation error after {seconds:.1f}s: {type(error).__name__}: {error}')
            raise
        seconds = time.monotonic() - started
        self._report(stage='execution', generation_s=seconds,
                    generated_tokens=sum(output.output_tokens for output in result))
        self._log(f'[llm] generation finished call={self.calls} requests={len(result)} seconds={seconds:.1f}')
        return result


## Cài đặt và kiểm tra môi trường

In [ ]:
# Thư viện cài thêm: ưu tiên wheel từ output notebook `build-wheels` (Add Input), không có / lỗi thì cài từ mạng
WHEEL_DIRS = sorted({str(Path(p).parent) for p in glob.glob("/kaggle/input/**/wheels/*.whl", recursive=True)})
print("wheel dirs:", WHEEL_DIRS or "none -> install from internet")

def pip_install(pkgs: str) -> int:
    if WHEEL_DIRS:
        links = " ".join(f"--find-links {d}" for d in WHEEL_DIRS)
        if sh(f"{sys.executable} -m pip install -q --no-index {links} {pkgs}") == 0:
            print("installed from wheels:", pkgs)
            return 0
        print("wheel install failed -> internet:", pkgs)
    return sh(f"{sys.executable} -m pip install -q {pkgs}")

pip_install("python-calamine")
# vLLM goes into its own venv: `pip install vllm` into the Kaggle env replaces torch (2.13+cu130 on the 2026-10 image)
# and breaks torchaudio -> transformers ASR. The base env stays untouched for ASR and the transformers fallback.
LLM_PYTHON = sys.executable
if CONFIG.get("llm", {}).get("backend", "auto") in ("auto", "vllm") and CONFIG.get("install_vllm", True):
    VENV = Path("/tmp/venv_vllm")  # not under /kaggle/working: it would be saved as output (GBs)
    pip_install("uv")
    sh(f"{sys.executable} -m uv venv -q {VENV} --python {sys.executable}")
    vpy = VENV / "bin" / "python"
    pkgs = f"{CONFIG.get('vllm_spec', 'vllm==0.32.0 wrapt')} pandas pyarrow openpyxl python-calamine"
    rc = 1
    if WHEEL_DIRS:
        links = " ".join(f"--find-links {d}" for d in WHEEL_DIRS)
        rc = sh(f"{sys.executable} -m uv pip install -q --python {vpy} --no-index {links} {pkgs}")
    if rc != 0:
        rc = sh(f"{sys.executable} -m uv pip install -q --python {vpy} {pkgs}")
    check = "import vllm, torch; assert torch.cuda.is_available(); print('vllm', vllm.__version__, 'torch', torch.__version__)"
    if rc == 0 and sh(f'{vpy} -c "{check}"') == 0:
        LLM_PYTHON = str(vpy)
print("LLM worker python:", LLM_PYTHON, "(transformers fallback)" if LLM_PYTHON == sys.executable else "(vLLM venv)")

In [ ]:
# Dataset: dùng bản đã Add Input trong /kaggle/input; nếu chưa có thì tự tải bằng kagglehub
# (https://www.kaggle.com/datasets/thanhkhil/zaloai-datamate-dataset)
DATASET = "thanhkhil/zaloai-datamate-dataset"
if not glob.glob(f"/kaggle/input/**/{CONFIG['split']}/questions.jsonl", recursive=True):
    pip_install("kagglehub")
    import kagglehub
    DATA_PATH = kagglehub.dataset_download(DATASET)
    print("dataset downloaded to", DATA_PATH)
    CONFIG["input_roots"] = [str(DATA_PATH)] + CONFIG["input_roots"]
else:
    print("dataset found under /kaggle/input")

In [ ]:
if CONFIG.get("strict_dev"):
    from dmlib.experiments import preset
    CONFIG = preset(CONFIG, CONFIG["preset"])
    CONFIG.update(DEV_OPTIONS)
    CONFIG["session_started_epoch"] = SESSION_STARTED_EPOCH
    CONFIG["session_deadline_epoch"] = SESSION_STARTED_EPOCH + CONFIG["session_wallclock_s"]
    # Set these paths explicitly for a continuation or an attached control/preflight output.
    # CONFIG["execution_id"] = "<next-run-id>"
    # CONFIG["resume_from_dirs"] = ["/kaggle/input/<previous-output>/<experiment-output-root>"]
    # CONFIG["preflight_dirs"] = ["/kaggle/input/<saved-preflight>/<experiment-output-root>"]
    # CONFIG["control_dir"] = "/kaggle/input/<control-output>/<experiment-output-root>"
import importlib
for m in list(sys.modules):
    if m == "dmlib" or m.startswith("dmlib."):
        del sys.modules[m]
from dmlib.common import env_info, find_split_dir, atomic_write_json
info = env_info(); print(json.dumps(info, indent=1, ensure_ascii=False))
N_GPUS = len(info.get("gpus", []))
SPLIT_DIR = find_split_dir(CONFIG["split"], CONFIG["input_roots"], marker="questions.jsonl")
print("split dir:", SPLIT_DIR, "| shops:", len(list((SPLIT_DIR / "data").iterdir())), "| has answers:", (SPLIT_DIR / "answers.jsonl").exists())
atomic_write_json(OUT / "environment.json", info)
if CONFIG.get("strict_dev"):
    import unittest
    suite = unittest.defaultTestLoader.loadTestsFromName("dmlib.regression")
    result = unittest.TextTestRunner(verbosity=1).run(suite)
    assert result.wasSuccessful(), "CPU regression failed; model preflight has not started"
    print("REGRESSION_OK", result.testsRun)

WHEELS_SHA256 = None
if WHEEL_DIRS:
    import hashlib
    manifest = []
    for directory in WHEEL_DIRS:
        d = Path(directory)
        manifest.append({"wheels": sorted((p.name, p.stat().st_size) for p in d.glob("*.whl")),
                         "metadata": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in d.iterdir()
                                      if p.name in ("SMOKE.json", "versions.json", "requirements.txt", "requirements-vllm.txt")}})
    WHEELS_SHA256 = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
    atomic_write_json(OUT / "wheels_manifest.json", {"sha256":WHEELS_SHA256,"sources":WHEEL_DIRS,"manifest":manifest})


In [ ]:
if not CONFIG.get("strict_dev") and CONFIG["mode"] == "helper":
    from dmlib.experiments import prepare_caches
    shops = sorted(p.name for p in (SPLIT_DIR / "data").iterdir() if p.is_dir())
    prepare_caches(CONFIG, SPLIT_DIR, shops)


In [ ]:
if not CONFIG.get("strict_dev"):
    # Resume: attach the output of an earlier version of this notebook as input; its shard files are copied back here
    # (same config only: a different config hash is refused by the worker).
    for d in CONFIG.get("resume_from_dirs", []):
        for p in glob.glob(f"{d}/**/preds.shard*.jsonl", recursive=True) + glob.glob(f"{d}/**/traces.shard*.jsonl", recursive=True):
            dst = OUT / Path(p).name
            if not dst.exists():
                shutil.copy(p, dst); print("resume <-", p)


## Chạy (một worker / GPU, shard theo id ổn định, ghi bền từng câu)

In [ ]:
if CONFIG.get("strict_dev"):
    from dmlib.experiments import run_dev
    rep = run_dev(CONFIG, SPLIT_DIR, LIB, list(range(N_GPUS)), LLM_PYTHON, WHEELS_SHA256)
    cfg = CONFIG
    print(json.dumps(rep, ensure_ascii=False, indent=1))
else:
    from dmlib.run import launch, select_items
    from dmlib.merge import merge
    cfg = dict(CONFIG, split_dir=str(SPLIT_DIR), num_shards=max(N_GPUS, 1))
    cfg_path = OUT / "config.json"; atomic_write_json(cfg_path, cfg)
    t0 = time.time()
    codes = launch(cfg, str(cfg_path), gpus=list(range(N_GPUS)), lib_root=str(LIB), python=LLM_PYTHON)
    print("exit codes:", codes, f"{(time.time() - t0) / 60:.1f} min")
    for log in sorted(OUT.glob("worker*.log")):
        print("=====", log.name); print(log.read_text(encoding="utf-8")[-2500:])


In [ ]:
if not CONFIG.get("strict_dev"):
    ids = [it.id for it in select_items(cfg, SPLIT_DIR)]
    rep = merge(cfg["out_dir"], SPLIT_DIR, finalize=CONFIG["finalize"], ids=ids)
    print(json.dumps({k: v for k, v in rep.items() if k not in ("by_shop",)}, ensure_ascii=False, indent=1))
    if rep.get("by_shop"):
        for s, v in rep["by_shop"].items(): print(s, v)
    answered = rep["n_questions"] - rep["pending"]
    if rep.get("submission") and CONFIG["split"] == "public_test":
        if answered == 0:
            # every line would be an abstain filled in for a crash: 0 points and a wasted daily submission
            print("NOT exporting /kaggle/working/submission.jsonl: no question was processed (see worker logs above)")
        else:
            shutil.copy(rep["submission"], WORK / "submission.jsonl")
            print(f"-> /kaggle/working/submission.jsonl ({answered}/{rep['n_questions']} processed, rest abstain)")
    for p in sorted(OUT.glob("summary.shard*.json")):
        print(p.name, json.loads(p.read_text())["s_per_item"], "s/question")


In [ ]:
# A few traces to eyeball (question, code, error, final answer)
tr = [json.loads(l) for p in sorted(OUT.glob("traces.shard*.jsonl")) for l in open(p, encoding="utf-8").read().splitlines()[1:]]
for t in tr[:3]:
    print("Q:", t["question"], "| as_of", t["as_of"]); a = t["attempts"][-1]
    print(a.get("code")); print("error:", (a.get("error") or "")[-500:]); print("final:", t["final"]); print("-" * 80)